In [1]:
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
import deepof.model_utils
import deepof.clustering.model_utils_new

# Test parameters matching the full model
batch_size = 2
time_steps = 25
n_nodes = 11
n_edges = 11
features_per_node = 3
features_per_edge = 1
latent_dim = 8

# Create test data
np.random.seed(42)
test_nodes = np.random.randn(batch_size, n_nodes, time_steps, features_per_node).astype(np.float32)
test_edges = np.random.randn(batch_size, n_edges, time_steps, features_per_edge).astype(np.float32)

print(f"Test node data shape: {test_nodes.shape}")
print(f"Test edge data shape: {test_edges.shape}")

Test node data shape: (2, 11, 25, 3)
Test edge data shape: (2, 11, 25, 1)


In [ ]:
def permute_gru_weights(keras_weights):
    """Permutes GRU weights from Keras (z, r, n) to PyTorch (r, z, n) format."""
    W_ih, W_hh, B = keras_weights
    # Keras gate order: z, r, n (update, reset, new/candidate)
    W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
    W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)

    # PyTorch gate order: r, z, n (reset, update, new/candidate)
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)

    # Keras has two bias vectors (input-hidden and recurrent), which are concatenated in B
    B_ih, B_hh = B
    B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
    B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])

    return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt

def transfer_recurrent_decoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent decoder model.
    """
    # Find layers by type to avoid index issues
    bidi_layers = [l for l in tf_model.layers if isinstance(l, Bidirectional)]
    norm_layers = [l for l in tf_model.layers if isinstance(l, LayerNormalization)]
    conv_layers = [l for l in tf_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
    prob_dec_layer = next(l for l in tf_model.layers if isinstance(l, deepof.model_utils.ProbabilisticDecoder))

    # --- GRU 1 and Norm 1 ---
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(bidi_layers[0].forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1)
    pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(bidi_layers[0].backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1)
    pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)
    pt_model.norm1.weight.data = torch.from_numpy(norm_layers[0].get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm_layers[0].get_weights()[1])

    # --- GRU 2 and Norm 2 ---
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(bidi_layers[1].forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2)
    pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(bidi_layers[1].backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2)
    pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    pt_model.norm2.weight.data = torch.from_numpy(norm_layers[1].get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm_layers[1].get_weights()[1])

    # --- Conv1D and Norm 3 ---
    # TF Conv1D weights: (kernel_w, kernel_h, in_c, out_c) -> (5, 1, 4*ld, 2*ld)
    # PT Conv1d weights: (out_c, in_c, kernel_w)
    conv_weights_tf = conv_layers[0].get_weights()[0]
    pt_model.conv1d.weight.data = torch.from_numpy(conv_weights_tf).squeeze(1).permute(2, 1, 0)
    pt_model.norm3.weight.data = torch.from_numpy(norm_layers[2].get_weights()[0]); pt_model.norm3.bias.data = torch.from_numpy(norm_layers[2].get_weights()[1])

    # --- Probabilistic Decoder ---
    # TF Dense weights: (in_features, out_features)
    # PT Linear weights: (out_features, in_features)
    prob_dec_weights, prob_dec_bias = prob_dec_layer.time_distributer.get_weights()
    pt_model.prob_decoder.loc_projection.weight.data = torch.from_numpy(prob_dec_weights.T)
    pt_model.prob_decoder.loc_projection.bias.data = torch.from_numpy(prob_dec_bias)


import numpy as np
import torch
import tensorflow as tf
from tensorflow.keras.layers import TimeDistributed, Conv1D, Bidirectional, LayerNormalization, Dense


def _to_torch(x):
    return torch.from_numpy(np.asarray(x))


def _extract_conv1d_weight(conv_layer):
    # Keras Conv1D kernel: (k, in_c, out_c)
    w = conv_layer.get_weights()[0]
    if w.ndim == 3:
        # -> PyTorch Conv1d: (out_c, in_c, k)
        return np.transpose(w, (2, 1, 0))
    elif w.ndim == 4 and w.shape[1] == 1:
        # Sometimes shows as (k, 1, in_c, out_c)
        w = np.squeeze(w, axis=1)
        return np.transpose(w, (2, 1, 0))
    else:
        raise RuntimeError(f"Unexpected Conv1D kernel shape: {w.shape}")


def _extract_gru_raw_weights(gru_layer):
    """
    Return (W_ih, W_hh, B_ih, B_hh) from a Keras GRU layer, normalized to:
      W_ih: (in, 3*units), W_hh: (units, 3*units)
      B_ih: (3*units,), B_hh: (3*units,)
    Handles reset_after=True case where bias can be 2D (2, 3*units).
    """
    ws = gru_layer.get_weights()
    units = gru_layer.units

    if len(ws) == 3:
        W_ih, W_hh, B = ws
        # B can be:
        # - flat (2*3*units,)
        # - 2D (2, 3*units)
        # - flat (3*units,) if reset_after=False (rare)
        if B.ndim == 2 and B.shape[0] == 2 and B.shape[1] == 3 * units:
            B_ih, B_hh = B[0].copy(), B[1].copy()
        elif B.ndim == 1 and B.shape[0] == 2 * 3 * units:
            half = 3 * units
            B_ih, B_hh = B[:half].copy(), B[half:].copy()
        elif B.ndim == 1 and B.shape[0] == 3 * units:
            # No recurrent bias; provide zeros for recurrent bias to match PyTorch expectations
            B_ih, B_hh = B.copy(), np.zeros_like(B)
        else:
            raise RuntimeError(f"Unexpected GRU bias shape: {B.shape}")
    elif len(ws) == 4:
        W_ih, W_hh, B_ih, B_hh = ws
        # Ensure 1D
        B_ih = B_ih.reshape(-1).copy()
        B_hh = B_hh.reshape(-1).copy()
    else:
        raise RuntimeError(f"Unexpected number of GRU weights: {len(ws)}")

    return W_ih, W_hh, B_ih, B_hh


def _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units):
    """
    Keras gate order: z, r, n -> PyTorch: r, z, n.
    Use explicit slicing by units (no np.split to avoid shape issues).
    Returns:
      W_ih_pt: (3h, in), W_hh_pt: (3h, h), B_ih_pt: (3h,), B_hh_pt: (3h,)
    """
    # W_ih: (in, 3*units)
    W_ih_z = W_ih[:, 0 * units:1 * units]
    W_ih_r = W_ih[:, 1 * units:2 * units]
    W_ih_n = W_ih[:, 2 * units:3 * units]
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1).T  # -> (3*units, in)

    # W_hh: (units, 3*units)
    W_hh_z = W_hh[:, 0 * units:1 * units]
    W_hh_r = W_hh[:, 1 * units:2 * units]
    W_hh_n = W_hh[:, 2 * units:3 * units]
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1).T  # -> (3*units, units)

    # Biases: (3*units,)
    B_ih_z = B_ih[0 * units:1 * units]
    B_ih_r = B_ih[1 * units:2 * units]
    B_ih_n = B_ih[2 * units:3 * units]
    B_hh_z = B_hh[0 * units:1 * units]
    B_hh_r = B_hh[1 * units:2 * units]
    B_hh_n = B_hh[2 * units:3 * units]

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])  # (3*units,)
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])  # (3*units,)

    return W_ih_pt, W_hh_pt, B_ih_pt, B_hh_pt


def transfer_recurrent_block_weights(tf_block_model, pt_block):
    """
    Minimal transfer for a single recurrent block (node or edge):
      - TimeDistributed(Conv1D)
      - TimeDistributed(Bidirectional(GRU)) x 2
      - LayerNormalization x 2
    """
    # Conv1D (inside TimeDistributed)
    td_convs = [l for l in tf_block_model.layers if isinstance(l, TimeDistributed) and isinstance(l.layer, Conv1D)]
    assert len(td_convs) >= 1, "No TimeDistributed(Conv1D) found in recurrent block"
    conv_td = td_convs[0]
    conv_w_pt = _extract_conv1d_weight(conv_td.layer)
    pt_block.conv1d.weight.data = _to_torch(conv_w_pt)
    if len(conv_td.layer.get_weights()) > 1 and pt_block.conv1d.bias is not None:
        pt_block.conv1d.bias.data = _to_torch(conv_td.layer.get_weights()[1])

    # Two TimeDistributed(Bidirectional(GRU))
    td_bis = [l for l in tf_block_model.layers if isinstance(l, TimeDistributed) and isinstance(l.layer, Bidirectional)]
    assert len(td_bis) >= 2, f"Expected 2 TimeDistributed(Bidirectional) GRUs, got {len(td_bis)}"
    bi1, bi2 = td_bis[0].layer, td_bis[1].layer

    # Norm layers
    norms = [l for l in tf_block_model.layers if isinstance(l, LayerNormalization)]
    assert len(norms) >= 2, f"Expected 2 LayerNormalization layers, got {len(norms)}"

    # First BiGRU -> pt_block.gru1 + norm1
    units1 = bi1.forward_layer.units
    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi1.forward_layer)
    W_ih_f, W_hh_f, B_ih_f, B_hh_f = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units1)
    pt_block.gru1.weight_ih_l0.data = _to_torch(W_ih_f)
    pt_block.gru1.weight_hh_l0.data = _to_torch(W_hh_f)
    pt_block.gru1.bias_ih_l0.data = _to_torch(B_ih_f)
    pt_block.gru1.bias_hh_l0.data = _to_torch(B_hh_f)

    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi1.backward_layer)
    W_ih_b, W_hh_b, B_ih_b, B_hh_b = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units1)
    pt_block.gru1.weight_ih_l0_reverse.data = _to_torch(W_ih_b)
    pt_block.gru1.weight_hh_l0_reverse.data = _to_torch(W_hh_b)
    pt_block.gru1.bias_ih_l0_reverse.data = _to_torch(B_ih_b)
    pt_block.gru1.bias_hh_l0_reverse.data = _to_torch(B_hh_b)

    pt_block.norm1.weight.data = _to_torch(norms[0].get_weights()[0])
    pt_block.norm1.bias.data = _to_torch(norms[0].get_weights()[1])

    # Second BiGRU -> pt_block.gru2 + norm2
    units2 = bi2.forward_layer.units
    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi2.forward_layer)
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units2)
    pt_block.gru2.weight_ih_l0.data = _to_torch(W_ih_f2)
    pt_block.gru2.weight_hh_l0.data = _to_torch(W_hh_f2)
    pt_block.gru2.bias_ih_l0.data = _to_torch(B_ih_f2)
    pt_block.gru2.bias_hh_l0.data = _to_torch(B_hh_f2)

    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi2.backward_layer)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units2)
    pt_block.gru2.weight_ih_l0_reverse.data = _to_torch(W_ih_b2)
    pt_block.gru2.weight_hh_l0_reverse.data = _to_torch(W_hh_b2)
    pt_block.gru2.bias_ih_l0_reverse.data = _to_torch(B_ih_b2)
    pt_block.gru2.bias_hh_l0_reverse.data = _to_torch(B_hh_b2)

    pt_block.norm2.weight.data = _to_torch(norms[1].get_weights()[0])
    pt_block.norm2.bias.data = _to_torch(norms[1].get_weights()[1])


def transfer_censnet_weights(tf_layer, pt_layer):
    """
    Spektral CensNetConv -> CensNetConvPT:
      Expected TF order: node_kernel, edge_kernel, node_weights, edge_weights, node_bias, edge_bias
    """
    kn_tf, ke_tf, pn_tf, pe_tf, bn_tf, be_tf = tf_layer.get_weights()
    pt_layer.node_kernel.data = _to_torch(kn_tf)
    pt_layer.edge_kernel.data = _to_torch(ke_tf)

    pn_tf = pn_tf.reshape(-1, 1) if pn_tf.ndim == 1 else pn_tf
    pe_tf = pe_tf.reshape(-1, 1) if pe_tf.ndim == 1 else pe_tf
    pt_layer.node_weights.data = _to_torch(pn_tf)
    pt_layer.edge_weights.data = _to_torch(pe_tf)

    pt_layer.node_bias.data = _to_torch(bn_tf)
    pt_layer.edge_bias.data = _to_torch(be_tf)


def _flatten_keras_tensors(x):
    if isinstance(x, (list, tuple)):
        out = []
        for xi in x:
            out.extend(_flatten_keras_tensors(xi))
        return out
    return [x]

def _layer_from_tensor(t):
    kh = getattr(t, "_keras_history", None)
    return kh[0] if isinstance(kh, tuple) else getattr(kh, "layer", None)

def transfer_recurrent_encoder_weights(tf_model, pt_model, verbose=False):
    try:
        tf_enc = tf_model.get_layer("recurrent_encoder")
    except Exception:
        tf_enc = tf_model

    # Final Dense
    w, b = tf_enc.layers[-1].get_weights()
    pt_model.final_dense.weight.data = torch.from_numpy(w.T)
    pt_model.final_dense.bias.data = torch.from_numpy(b)

    if not getattr(pt_model, "use_gnn", False):
        sub = [l for l in tf_enc.layers if isinstance(l, tf.keras.Model)][0]
        transfer_recurrent_block_weights(sub, pt_model.recurrent_block)
        return

    node_in_ch = int(pt_model.node_recurrent_block.conv1d.in_channels)
    edge_in_ch = int(pt_model.edge_recurrent_block.conv1d.in_channels)
    gnn_tf, _, _, tf_node_block, tf_edge_block = _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch)

    transfer_recurrent_block_weights(tf_node_block, pt_model.node_recurrent_block)
    transfer_recurrent_block_weights(tf_edge_block, pt_model.edge_recurrent_block)
    transfer_censnet_weights(gnn_tf, pt_model.spatial_gnn_block)

    if verbose:
        print(f"Transferred node from {tf_node_block.name} (in_ch={node_in_ch}), edge from {tf_edge_block.name} (in_ch={edge_in_ch})")


In [6]:
import deepof.clustering.models_new


def transfer_vq_weights(tf_model: deepof.models.VectorQuantizer, pt_model: deepof.clustering.models_new.VectorQuantizerPT):
    """Transfers weights from TensorFlow VectorQuantizer to PyTorch version.
    
    Args:
        tf_model: TensorFlow VectorQuantizer model
        pt_model: PyTorch VectorQuantizerPT model
    """
    # Transfer codebook weights
    # TF shape: (embedding_dim, n_components)
    # PT shape: (embedding_dim, n_components)
    # These match, so direct transfer
    codebook_weights = tf_model.codebook.numpy()
    pt_model.codebook.data = torch.from_numpy(codebook_weights)
    
    print(f"✅ Transferred codebook weights: {codebook_weights.shape}")

def transfer_vqvae_weights(tf_vqvae, pt_vqvae):
    """
    Transfer weights from TensorFlow VQVAE to PyTorch VQVAEPT.
    
    Args:
        tf_vqvae: TensorFlow VQVAE instance
        pt_vqvae: PyTorch VQVAEPT instance
    """
    print("Transferring VQ-VAE weights...")
    
    # Transfer encoder weights (assumed already done via encoder-specific transfer functions)
    # transfer_encoder_weights(tf_vqvae.encoder, pt_vqvae.encoder)
    print("  ⚠️  Encoder weights - use encoder-specific transfer function")
    
    # Transfer VQ layer weights
    tf_vq_layer = tf_vqvae.vqvae.get_layer('vector_quantizer')
    transfer_vq_weights(tf_vq_layer, pt_vqvae.vq_layer)
    print("  ✅ VQ layer weights transferred")
    
    # Transfer decoder weights (assumed already done via decoder-specific transfer functions)
    # transfer_decoder_weights(tf_vqvae.decoder, pt_vqvae.decoder)
    print("  ⚠️  Decoder weights - use decoder-specific transfer function")
    
    print("✅ VQ-VAE weight transfer complete!")

In [5]:
# Create TF recurrent block for nodes
tf_node_input = tf.keras.Input(shape=(n_nodes, time_steps, features_per_node))
tf_node_recurrent = deepof.model_utils.get_recurrent_block(
    tf_node_input, 
    latent_dim, 
    gru_unroll=False, 
    bidirectional_merge="concat"
)(tf_node_input)
tf_node_model = tf.keras.Model(inputs=tf_node_input, outputs=tf_node_recurrent)

# Create TF recurrent block for edges
tf_edge_input = tf.keras.Input(shape=(n_edges, time_steps, features_per_edge))
tf_edge_recurrent = deepof.model_utils.get_recurrent_block(
    tf_edge_input,
    latent_dim,
    gru_unroll=False,
    bidirectional_merge="concat"
)(tf_edge_input)
tf_edge_model = tf.keras.Model(inputs=tf_edge_input, outputs=tf_edge_recurrent)

# Test forward pass
tf_node_output = tf_node_model(test_nodes, training=False)
tf_edge_output = tf_edge_model(test_edges, training=False)

print(f"TF node model output shape: {tf_node_output.shape}")
print(f"TF edge model output shape: {tf_edge_output.shape}")
print(f"TF node output sample: {tf_node_output[0, 0, :5].numpy()}")
print(f"TF edge output sample: {tf_edge_output[0, 0, :5].numpy()}")

# Display model structure
print("\nTF Node Model layers:")
for i, layer in enumerate(tf_node_model.layers):
    print(f"  {i}: {layer.name} - {type(layer).__name__}")
    if hasattr(layer, 'layers'):  # If it's a nested model
        for j, sublayer in enumerate(layer.layers):
            print(f"    {j}: {sublayer.name} - {type(sublayer).__name__}")

TF node model output shape: (2, 11, 16)
TF edge model output shape: (2, 11, 16)
TF node output sample: [-0.18209487  1.2535563   0.6659341   1.3584192   1.3101838 ]
TF edge output sample: [ 1.2105745e+00  3.3642778e-01  1.5770061e+00 -5.5604172e-04
 -1.4515566e+00]

TF Node Model layers:
  0: input_1 - InputLayer
  1: model - Functional
    0: input_1 - InputLayer
    1: time_distributed - TimeDistributed
    2: masking - Masking
    3: time_distributed_1 - TimeDistributed
    4: layer_normalization - LayerNormalization
    5: time_distributed_2 - TimeDistributed
    6: layer_normalization_1 - LayerNormalization


In [6]:
# Create PyTorch recurrent blocks
pt_node_recurrent = deepof.clustering.model_utils_new.RecurrentBlockPT(
    input_features=features_per_node, 
    latent_dim=latent_dim
)

pt_edge_recurrent = deepof.clustering.model_utils_new.RecurrentBlockPT(
    input_features=features_per_edge,
    latent_dim=latent_dim
)



# Transfer weights
transfer_recurrent_block_weights(tf_node_model.layers[-1], pt_node_recurrent)
transfer_recurrent_block_weights(tf_edge_model.layers[-1], pt_edge_recurrent)

In [7]:
def compare_recurrent_outputs(tf_model, pt_model, test_data, block_name):
    """Compare TF and PT recurrent block outputs."""
    print(f"\n{'='*50}")
    print(f"Comparing {block_name}")
    print(f"{'='*50}")
    
    # TF forward pass
    tf_output = tf_model(test_data, training=False).numpy()
    
    # PT forward pass
    pt_model.eval()
    with torch.no_grad():
        pt_input = torch.from_numpy(test_data)
        pt_output = pt_model(pt_input).numpy()
    
    # Compare
    max_diff = np.abs(tf_output - pt_output).max()
    mean_diff = np.abs(tf_output - pt_output).mean()
    
    print(f"Output shapes: TF {tf_output.shape}, PT {pt_output.shape}")
    print(f"Max difference: {max_diff:.8f}")
    print(f"Mean difference: {mean_diff:.8f}")
    print(f"TF output sample: {tf_output[0, 0, :5]}")
    print(f"PT output sample: {pt_output[0, 0, :5]}")
    
    # Check if outputs match
    matches = np.allclose(tf_output, pt_output, rtol=1e-5, atol=1e-5)
    print(f"Outputs match: {matches}")
    
    return max_diff, tf_output, pt_output

test_nodes[0,0,0,:]=0.0
test_nodes[1,1,:,1]=0.0
test_nodes[1,:,2,2]=0.0
test_edges[0,0,0,:]=0.0
test_edges[1,1,:,0]=0.0
test_edges[1,:,2,0]=0.0

# Compare node recurrent block
node_diff, tf_node_out, pt_node_out = compare_recurrent_outputs(
    tf_node_model, pt_node_recurrent, test_nodes, "Node Recurrent Block"
)

# Compare edge recurrent block
edge_diff, tf_edge_out, pt_edge_out = compare_recurrent_outputs(
    tf_edge_model, pt_edge_recurrent, test_edges, "Edge Recurrent Block"
)

# Additional detailed comparison
if node_diff > 1e-5 or edge_diff > 1e-5:
    print("\n" + "="*50)
    print("Detailed difference analysis")
    print("="*50)
    
    if node_diff > 1e-5:
        node_diffs = np.abs(tf_node_out - pt_node_out)
        worst_idx = np.unravel_index(np.argmax(node_diffs), node_diffs.shape)
        print(f"Node worst difference at index {worst_idx}:")
        print(f"  TF value: {tf_node_out[worst_idx]}")
        print(f"  PT value: {pt_node_out[worst_idx]}")
        print(f"  Difference: {node_diffs[worst_idx]}")
    
    if edge_diff > 1e-5:
        edge_diffs = np.abs(tf_edge_out - pt_edge_out)
        worst_idx = np.unravel_index(np.argmax(edge_diffs), edge_diffs.shape)
        print(f"Edge worst difference at index {worst_idx}:")
        print(f"  TF value: {tf_edge_out[worst_idx]}")
        print(f"  PT value: {pt_edge_out[worst_idx]}")
        print(f"  Difference: {edge_diffs[worst_idx]}")


Comparing Node Recurrent Block
Output shapes: TF (2, 11, 16), PT (2, 11, 16)
Max difference: 0.00000066
Mean difference: 0.00000013
TF output sample: [-0.16907986  1.2758803   0.68413067  1.3812369   1.332848  ]
PT output sample: [-0.16907997  1.2758803   0.6841305   1.3812369   1.332848  ]
Outputs match: True

Comparing Edge Recurrent Block
Output shapes: TF (2, 11, 16), PT (2, 11, 16)
Max difference: 0.00000083
Mean difference: 0.00000012
TF output sample: [ 1.252241    0.31836933  1.6439373  -0.04645128 -1.5955242 ]
PT output sample: [ 1.2522411   0.3183693   1.6439375  -0.04645133 -1.5955244 ]
Outputs match: True


In [21]:
import numpy as np
import torch
import tensorflow as tf
from spektral.layers import CensNetConv
from deepof.clustering.censNetConv_pt import CensNetConvPT

# Create a simple adjacency matrix for testing
def create_test_adjacency(n_nodes=4):
    """Create a simple test adjacency matrix."""
    adj = np.zeros((n_nodes, n_nodes), dtype=np.float32)  # Ensure float32
    # Create a simple connected graph
    for i in range(n_nodes - 1):
        adj[i, i+1] = 1
        adj[i+1, i] = 1
    # Add one more edge to make it interesting
    if n_nodes > 2:
        adj[0, n_nodes-1] = 1
        adj[n_nodes-1, 0] = 1
    return adj

# Test parameters
n_nodes = 4
n_edges = 4  # For our simple graph
batch_size = 2
input_node_features = 8
input_edge_features = 8
output_channels = 4

adj_matrix = create_test_adjacency(n_nodes)
print("Adjacency matrix:")
print(adj_matrix)

Adjacency matrix:
[[0. 1. 0. 1.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [1. 0. 1. 0.]]


In [22]:
# Create TensorFlow CensNetConv layer
tf_layer = CensNetConv(
    node_channels=output_channels,
    edge_channels=output_channels,
    activation='relu'
)

# Preprocess adjacency for TF
tf_lap, tf_edge_lap, tf_inc = tf_layer.preprocess(adj_matrix)
print(f"TF Laplacian shape: {tf_lap.shape}")
print(f"TF Edge Laplacian shape: {tf_edge_lap.shape}")
print(f"TF Incidence shape: {tf_inc.shape}")

# Create test inputs for TF
tf_node_features = tf.random.normal((batch_size, n_nodes, input_node_features))
tf_edge_features = tf.random.normal((batch_size, n_edges, input_edge_features))

# Build the layer by calling it once
tf_output = tf_layer([tf_node_features, (tf_lap, tf_edge_lap, tf_inc), tf_edge_features])
print(f"\nTF output shapes: nodes={tf_output[0].shape}, edges={tf_output[1].shape}")

# Get weights
tf_weights = tf_layer.get_weights()
print(f"\nTF layer weights count: {len(tf_weights)}")
for i, w in enumerate(tf_weights):
    print(f"  Weight {i}: shape {w.shape}")

TF Laplacian shape: (4, 4)
TF Edge Laplacian shape: (4, 4)
TF Incidence shape: (4, 4)

TF output shapes: nodes=(2, 4, 4), edges=(2, 4, 4)

TF layer weights count: 6
  Weight 0: shape (8, 4)
  Weight 1: shape (8, 4)
  Weight 2: shape (8, 1)
  Weight 3: shape (8, 1)
  Weight 4: shape (4,)
  Weight 5: shape (4,)


The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.


In [23]:
# Create PyTorch CensNetConvPT layer
pt_layer = CensNetConvPT(
    node_channels=output_channels,
    edge_channels=output_channels,
    activation='relu'
)

# Initialize with correct dimensions
pt_layer._build(
    node_features_shape=[batch_size, n_nodes, input_node_features],
    edge_features_shape=[batch_size, n_edges, input_edge_features]
)

transfer_censnet_weights(tf_layer, pt_layer)

# Preprocess adjacency for PT - ensure float32
pt_lap, pt_edge_lap, pt_inc = pt_layer.preprocess(torch.tensor(adj_matrix, dtype=torch.float32))
print(f"\nPT Laplacian shape: {pt_lap.shape}")
print(f"PT Edge Laplacian shape: {pt_edge_lap.shape}")
print(f"PT Incidence shape: {pt_inc.shape}")


PT Laplacian shape: torch.Size([4, 4])
PT Edge Laplacian shape: torch.Size([4, 4])
PT Incidence shape: torch.Size([4, 4])


In [24]:
def compare_outputs(tf_layer, pt_layer, test_name, tf_nodes, tf_edges, pt_nodes, pt_edges, 
                    tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc):
    """Compare TF and PT outputs for given inputs."""
    print(f"\n{'='*50}")
    print(f"Test: {test_name}")
    print(f"{'='*50}")
    
    # Ensure all PT tensors are float32
    pt_nodes = pt_nodes.float()
    pt_edges = pt_edges.float()
    pt_lap = pt_lap.float()
    pt_edge_lap = pt_edge_lap.float()
    pt_inc = pt_inc.float()
    
    # TF forward pass
    tf_out_nodes, tf_out_edges = tf_layer([tf_nodes, (tf_lap, tf_edge_lap, tf_inc), tf_edges])
    
    # PT forward pass
    pt_layer.eval()
    with torch.no_grad():
        pt_out_nodes, pt_out_edges = pt_layer([pt_nodes, (pt_lap, pt_edge_lap, pt_inc), pt_edges])
    
    # Convert to numpy for comparison
    tf_out_nodes_np = tf_out_nodes.numpy()
    tf_out_edges_np = tf_out_edges.numpy()
    pt_out_nodes_np = pt_out_nodes.numpy()
    pt_out_edges_np = pt_out_edges.numpy()
    
    # Compare
    node_diff = np.abs(tf_out_nodes_np - pt_out_nodes_np).max()
    edge_diff = np.abs(tf_out_edges_np - pt_out_edges_np).max()
    
    print(f"Node output diff: {node_diff:.8f}")
    print(f"Edge output diff: {edge_diff:.8f}")
    print(f"TF node output sample: {tf_out_nodes_np[0, 0, :3]}")
    print(f"PT node output sample: {pt_out_nodes_np[0, 0, :3]}")
    print(f"TF edge output sample: {tf_out_edges_np[0, 0, :3]}")
    print(f"PT edge output sample: {pt_out_edges_np[0, 0, :3]}")
    
    return node_diff, edge_diff

# Test 1: Random inputs
print("Test 1: Random inputs")
tf_nodes_random = tf.random.normal((batch_size, n_nodes, input_node_features), seed=42)
tf_edges_random = tf.random.normal((batch_size, n_edges, input_edge_features), seed=43)
pt_nodes_random = torch.from_numpy(tf_nodes_random.numpy()).float()
pt_edges_random = torch.from_numpy(tf_edges_random.numpy()).float()

compare_outputs(tf_layer, pt_layer, "Random inputs", 
                tf_nodes_random, tf_edges_random, pt_nodes_random, pt_edges_random,
                tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc)

# Test 2: Simple inputs (ones)
print("\nTest 2: Ones")
tf_nodes_ones = tf.ones((batch_size, n_nodes, input_node_features))
tf_edges_ones = tf.ones((batch_size, n_edges, input_edge_features))
pt_nodes_ones = torch.ones((batch_size, n_nodes, input_node_features), dtype=torch.float32)
pt_edges_ones = torch.ones((batch_size, n_edges, input_edge_features), dtype=torch.float32)

compare_outputs(tf_layer, pt_layer, "Ones", 
                tf_nodes_ones, tf_edges_ones, pt_nodes_ones, pt_edges_ones,
                tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc)

# Test 3: Identity-like pattern
print("\nTest 3: Identity pattern")
tf_nodes_eye = tf.constant(np.eye(input_node_features, dtype=np.float32)[:n_nodes], dtype=tf.float32)
tf_nodes_eye = tf.expand_dims(tf_nodes_eye, 0)
tf_nodes_eye = tf.tile(tf_nodes_eye, [batch_size, 1, 1])
tf_edges_eye = tf.constant(np.eye(input_edge_features, dtype=np.float32)[:n_edges], dtype=tf.float32)
tf_edges_eye = tf.expand_dims(tf_edges_eye, 0)
tf_edges_eye = tf.tile(tf_edges_eye, [batch_size, 1, 1])

pt_nodes_eye = torch.from_numpy(tf_nodes_eye.numpy()).float()
pt_edges_eye = torch.from_numpy(tf_edges_eye.numpy()).float()

compare_outputs(tf_layer, pt_layer, "Identity pattern", 
                tf_nodes_eye, tf_edges_eye, pt_nodes_eye, pt_edges_eye,
                tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc)

# Check if preprocessing outputs match
print("\n" + "="*50)
print("Checking preprocessing outputs:")
print("="*50)
print(f"Laplacian match: {np.allclose(tf_lap, pt_lap.numpy())}")
print(f"Edge Laplacian match: {np.allclose(tf_edge_lap, pt_edge_lap.numpy())}")
print(f"Incidence match: {np.allclose(tf_inc, pt_inc.numpy())}")
print(f"Laplacian diff: {np.abs(tf_lap - pt_lap.numpy()).max():.8f}")
print(f"Edge Laplacian diff: {np.abs(tf_edge_lap - pt_edge_lap.numpy()).max():.8f}")
print(f"Incidence diff: {np.abs(tf_inc - pt_inc.numpy()).max():.8f}")

Test 1: Random inputs

Test: Random inputs
DEBUG _propagate_nodes: node_features shape: torch.Size([2, 4, 8])
DEBUG _propagate_nodes: edge_features shape: torch.Size([2, 4, 8])
DEBUG _propagate_nodes: self.node_kernel shape: torch.Size([8, 4])
DEBUG _propagate_nodes: self.edge_weights shape: torch.Size([8, 1])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 1])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 4]), b.shape=torch.Size([4, 4])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 4])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 1])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 4]), b.shape=torch.Size([4, 4])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 4])
Node output diff: 0.00000000
Edge output diff: 0.00000000
TF node output sample: [0.19316557 0

# Vector quantizer test

In [6]:
from typing import Any, NewType, Iterable, Tuple
import unittest
import numpy as np
import tcn
import sys
import tensorflow as tf
import tensorflow_probability as tfp
from sklearn.mixture import GaussianMixture
from spektral.layers import CensNetConv
from tensorflow.keras import Input, Model
from tensorflow.keras.initializers import he_uniform
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)
from tensorflow.keras.optimizers import Nadam
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.distributions import Distribution, TransformedDistribution, Normal
from torch.distributions.transforms import AffineTransform
import time

import deepof.model_utils
import deepof.clustering.model_utils_new
from deepof.clustering.censNetConv_pt import CensNetConvPT
import deepof.utils
from deepof.data_loading import get_dt
import warnings
from deepof.clustering.model_utils_new import ProbabilisticDecoderPT

In [7]:
import deepof.clustering.models_new


class TestRecurrentEncoderTranslation(unittest.TestCase):
    def setUp(self):
        """Set up parameters and create random data that matches model assumptions."""
        tf.keras.backend.clear_session()
        self.latent_dim = 8
        
        # Real data configuration
        self.b = 2  # Batch
        self.t = 25  # Time
        self.n = 1  # Nodes  
        self.f_per_node = 1  # Features per node
        self.f_total = self.n * self.f_per_node  # Total features (33)
        self.e = 1  # Edges
        self.f_edge_per_edge = 1  # Features per edge
        self.f_edge_total = self.e * self.f_edge_per_edge  # Total edge features

        # For encoder, use FLATTENED shapes (as VQVAE does)
        self.input_shape = (self.t, self.f_total)  # (25, 33)
        self.edge_shape = (self.t, self.f_edge_total)  # (25, 11)
        # Create dummy adjacency matrix
        m = np.zeros((self.n, self.n))
        ui = np.triu_indices(self.n)
        num_possible_edges = len(ui[0])
        c = np.random.choice(num_possible_edges, min(self.e, num_possible_edges), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        m += m.T # Make symmetric
        self.adj_matrix = m

        # Check actual edge count
        actual_edge_count = np.sum(np.triu(self.adj_matrix) > 0)
        print(f"Expected edges: {self.e}, Actual edges in adjacency: {actual_edge_count}")
        print(f"Incidence matrix will have shape: (nodes={self.n}, edges={actual_edge_count})")

        # Create random input data in FLATTENED format
        self.x_np = np.random.rand(self.b, self.t, self.f_total).astype(np.float32)
        self.a_np = np.random.rand(self.b, self.t, self.f_edge_total).astype(np.float32)
        
        # For PyTorch, we need to reshape to (B, T, N, F) format
        self.x_np_pt = self.x_np.reshape(self.b, self.t, self.n, self.f_per_node)
        self.a_np_pt = self.a_np.reshape(self.b, self.t, self.e, self.f_edge_per_edge)
        
    def test_forward_pass_gnn(self):
        """Test the GNN-enabled path of the encoder."""
        # Build TF model with FLATTENED input shapes
        tf_model_gnn = deepof.models.get_recurrent_encoder(
            self.input_shape,  # (25, 33)
            self.edge_shape,   # (25, 11)
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=True
        )
        # After creating tf_model_gnn, check the incidence matrix shape
        tf_cens_layer = tf_model_gnn.get_layer('cens_net_conv')
        tf_lap, tf_edge_lap, tf_inc = tf_cens_layer.preprocess(self.adj_matrix)
        print(f"\nTF preprocessing shapes:")
        print(f"  TF Incidence shape: {tf_inc.shape}")
        print(f"  TF Edge Laplacian shape: {tf_edge_lap.shape}")

        # Debug TF model structure
        print("\nDEBUG: TensorFlow model layers:")
        for i, layer in enumerate(tf_model_gnn.layers):
            print(f"  Layer {i}: {layer.name} - {type(layer).__name__}")
            if hasattr(layer, 'output_shape'):
                print(f"    Output shape: {layer.output_shape}")
            if hasattr(layer, 'get_weights'):
                weights = layer.get_weights()
                if weights:
                    print(f"    Weight shapes: {[w.shape for w in weights]}")
        
        # Build PT model with UNFLATTENED input shapes 
        pt_input_shape = (self.t, self.n, self.f_per_node)  # (25, 11, 3)
        pt_edge_shape = (self.t, self.e, self.f_edge_per_edge)  # (25, 11, 1)
        
        pt_model_gnn = deepof.clustering.models_new.RecurrentEncoderPT(
            pt_input_shape, 
            pt_edge_shape, 
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=True
        )
        pt_model_gnn.eval()

        print(f"\nPT preprocessing shapes:")
        print(f"  PT Incidence shape: {pt_model_gnn.incidence.shape}")
        print(f"  PT Edge Laplacian shape: {pt_model_gnn.edge_laplacian.shape}")
        print(f"  PT num_edges attribute: {pt_model_gnn.num_edges}")

        # Run a single "dummy" forward pass on the PyTorch model to initialize weights
        #with torch.no_grad():
        #    pt_model_gnn(torch.from_numpy(self.x_np_pt), torch.from_numpy(self.a_np_pt))

        # Transfer weights from TF to PT
        transfer_recurrent_encoder_weights(tf_model_gnn, pt_model_gnn)

        # Execute and compare the outputs
        # TF gets flattened input
        tf_start = time.time()
        tf_output = tf_model_gnn([self.x_np, self.a_np], training=False).numpy()
        tf_end = time.time()
        
        # PT gets unflattened input
        pt_start = time.time()
        with torch.no_grad():
            pt_output = pt_model_gnn(
                torch.from_numpy(self.x_np_pt), 
                torch.from_numpy(self.a_np_pt)
            ).detach().numpy()
        pt_end = time.time()

        print(f"Tensorflow execution time: {tf_end-tf_start:.4f}s")
        print(f"Pytorch execution time: {pt_end-pt_start:.4f}s")
        print(f"TF output shape: {tf_output.shape}")
        print(f"PT output shape: {pt_output.shape}")
        print(f"TF output sample: {tf_output[0, :5]}")
        print(f"PT output sample: {pt_output[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_output - pt_output).max():.8f}")

        np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-4)
        print("✅ `RecurrentEncoderPT` (GNN path) translation test PASSED!")

    def test_forward_pass_no_gnn(self):
        """Test the non-GNN path of the encoder."""
        # Build TF model with FLATTENED input shapes
        tf_model_no_gnn = deepof.models.get_recurrent_encoder(
            self.input_shape,  # (25, 33)
            self.edge_shape,   # (25, 11)
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=False
        )
        
        # Build PT model with UNFLATTENED input shapes
        pt_input_shape = (self.t, self.n, self.f_per_node)  # (25, 11, 3)
        pt_edge_shape = (self.t, self.e, self.f_edge_per_edge)  # (25, 11, 1)
        
        pt_model_no_gnn = deepof.clustering.models_new.RecurrentEncoderPT(
            pt_input_shape, 
            pt_edge_shape, 
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=False
        )
        pt_model_no_gnn.eval()

        # Transfer weights
        transfer_recurrent_encoder_weights(tf_model_no_gnn, pt_model_no_gnn)

        # Execute and compare
        # TF gets flattened input
        tf_output = tf_model_no_gnn([self.x_np, self.a_np], training=False).numpy()
        
        # PT gets unflattened input
        with torch.no_grad():
            pt_output = pt_model_no_gnn(
                torch.from_numpy(self.x_np_pt), 
                torch.from_numpy(self.a_np_pt)
            ).detach().numpy()
        
        print(f"TF output shape: {tf_output.shape}")
        print(f"PT output shape: {pt_output.shape}")
        print(f"TF output sample: {tf_output[0, :5]}")
        print(f"PT output sample: {pt_output[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_output - pt_output).max():.8f}")

        np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-5)
        print("✅ `RecurrentEncoderPT` (non-GNN path) translation test PASSED!")

# To run:
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestRecurrentEncoderTranslation)
runner.run(suite)

test_forward_pass_gnn (__main__.TestRecurrentEncoderTranslation)
Test the GNN-enabled path of the encoder. ... 

Expected edges: 1, Actual edges in adjacency: 1
Incidence matrix will have shape: (nodes=1, edges=1)

TF preprocessing shapes:
  TF Incidence shape: (1, 1)
  TF Edge Laplacian shape: (1, 1)

DEBUG: TensorFlow model layers:
  Layer 0: input_1 - InputLayer
    Output shape: [(None, 25, 1)]
  Layer 1: input_2 - InputLayer
    Output shape: [(None, 25, 1)]
  Layer 2: tf.compat.v1.transpose - TFOpLambda
    Output shape: (1, 25, None)
  Layer 3: tf.compat.v1.transpose_2 - TFOpLambda
    Output shape: (1, 25, None)
  Layer 4: tf.reshape - TFOpLambda
    Output shape: (1, 25, 1, None)
  Layer 5: tf.reshape_1 - TFOpLambda
    Output shape: (1, 25, 1, None)
  Layer 6: tf.compat.v1.transpose_1 - TFOpLambda
    Output shape: (None, 1, 25, 1)
  Layer 7: tf.compat.v1.transpose_3 - TFOpLambda
    Output shape: (None, 1, 25, 1)
  Layer 8: model - Functional
    Output shape: (None, 1, 16)
    Weight shapes: [(5, 1, 16), (16, 48), (16, 48), (2, 48), (16, 48), (16, 48), (2, 48), (32,), (32,), (32, 24),

FAIL
test_forward_pass_no_gnn (__main__.TestRecurrentEncoderTranslation)
Test the non-GNN path of the encoder. ... 

Expected edges: 1, Actual edges in adjacency: 1
Incidence matrix will have shape: (nodes=1, edges=1)


ok

FAIL: test_forward_pass_gnn (__main__.TestRecurrentEncoderTranslation)
Test the GNN-enabled path of the encoder.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_15976\2578460083.py", line 121, in test_forward_pass_gnn
    np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-4)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\numpy\testing\_private\utils.py", line 1504, in assert_allclose
    assert_array_compare(compare, actual, desired, err_msg=str(err_msg),
  File "C:\Users\Petron\AppData\Local\Programs\Python\Python310\lib\contextlib.py", line 79, in inner
    return func(*args, **kwds)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\numpy\testing\_private\utils.py", line 797, in assert_array_compare
    raise AssertionError(msg)
AssertionError: 
Not equal to tolerance rtol=1e-05, atol=0.0001

Misma


DEBUG RecurrentEncoderPT forward:
  Input x shape: torch.Size([2, 25, 1, 1])
  Input a shape: torch.Size([2, 25, 1, 1])
  use_gnn: False
  Final output: torch.Size([2, 8])
TF output shape: (2, 8)
PT output shape: (2, 8)
TF output sample: [ 0.35755023  2.0263896  -1.8466074   0.04479006  2.5075285 ]
PT output sample: [ 0.3575502   2.0263898  -1.8466076   0.04479012  2.507528  ]
Max absolute difference: 0.00000095
✅ `RecurrentEncoderPT` (non-GNN path) translation test PASSED!


<unittest.runner.TextTestResult run=2 errors=0 failures=1>

In [8]:
class VectorQuantizer(tf.keras.models.Model):
    """Vector quantizer layer.

    Quantizes the input vectors into a fixed number of clusters using L2 norm. Based on
    https://arxiv.org/pdf/1509.03700.pdf, and adapted for clustering using https://arxiv.org/abs/1806.02199.
    Implementation based on https://keras.io/examples/generative/vq_vae/.

    """

    def __init__(
        self, n_components, embedding_dim, beta, kmeans_loss: float = 0.0, **kwargs
    ):
        """Initialize the VQ layer.

        Args:
            n_components (int): number of embeddings to use
            embedding_dim (int): dimensionality of the embeddings
            beta (float): beta value for the loss function
            kmeans_loss (float): regularization parameter for the Gram matrix
            **kwargs: additional arguments for the parent class

        """
        super(VectorQuantizer, self).__init__(**kwargs)
        self.embedding_dim = embedding_dim
        self.n_components = n_components
        self.beta = beta
        self.kmeans = kmeans_loss

        # Initialize the VQ codebook
        w_init = tf.random_uniform_initializer()
        self.codebook = tf.Variable(
            initial_value=w_init(
                shape=(self.embedding_dim, self.n_components), dtype="float32"
            ),
            trainable=True,
            name="vqvae_codebook",
        )

    def call(self, x):  # pragma: no cover
        """Compute the VQ layer.

        Args:
            x (tf.Tensor): input tensor

        Returns:
                x (tf.Tensor): output tensor
        """
        # Compute input shape and flatten, keeping the embedding dimension intact
        input_shape = tf.shape(x)

        # Add a disentangling penalty to the embeddings
        if self.kmeans:
            kmeans_loss = deepof.model_utils.compute_kmeans_loss(
                x, weight=self.kmeans, batch_size=input_shape[0]
            )
            self.add_loss(kmeans_loss)
            self.add_metric(kmeans_loss, name="kmeans_loss")

        flattened = tf.reshape(x, [-1, self.embedding_dim])

        # Quantize input using the codebook
        encoding_indices = tf.cast(
            self.get_code_indices(flattened, return_soft_counts=False), tf.int32
        )
        soft_counts = self.get_code_indices(flattened, return_soft_counts=True)

        encodings = tf.one_hot(encoding_indices, self.n_components)

        quantized = tf.matmul(encodings, self.codebook, transpose_b=True)
        quantized = tf.reshape(quantized, input_shape)

        # Compute vector quantization loss, and add it to the layer
        commitment_loss = self.beta * tf.reduce_mean(
            (tf.stop_gradient(quantized) - x) ** 2
        )
        codebook_loss = tf.reduce_mean((quantized - tf.stop_gradient(x)) ** 2)
        self.add_loss(commitment_loss + codebook_loss)

        # Straight-through estimator (copy gradients through the undiferentiable layer)
        # This approach has been reported to have issues for clustering, so we use add an extra
        # reconstruction loss to ensure that the gradients can flow through the encoder.
        # quantized = x + tf.stop_gradient(quantized - x)

        return quantized, soft_counts

    # noinspection PyTypeChecker
    def get_code_indices(
        self, flattened_inputs, return_soft_counts=False
    ):  # pragma: no cover
        """Getter for the code indices at any given time.

        Args:
            flattened_inputs (tf.Tensor): flattened input tensor (encoder output)
            return_soft_counts (bool): whether to return soft counts based on the distance to the codes, instead of the code indices

        Returns:
            encoding_indices (tf.Tensor): code indices tensor with cluster assignments.
        """
        # Compute L2-norm distance between inputs and codes at a given time
        similarity = tf.matmul(flattened_inputs, self.codebook)
        distances = (
            tf.reduce_sum(flattened_inputs**2, axis=1, keepdims=True)
            + tf.reduce_sum(self.codebook**2, axis=0)
            - 2 * similarity
        )

        if return_soft_counts:
            # Compute soft counts based on the distance to the codes
            similarity = (1 / distances) ** 2
            soft_counts = similarity / tf.expand_dims(
                tf.reduce_sum(similarity, axis=1), axis=1
            )
            return soft_counts

        # Return index of the closest code
        encoding_indices = tf.argmin(distances, axis=1)
        return encoding_indices

In [9]:
class VectorQuantizerPT(nn.Module):
    """PyTorch Vector quantizer layer."""

    def __init__(
        self,
        n_components: int,
        embedding_dim: int,
        beta: float,
        kmeans_loss: float = 0.0,
    ):
        super(VectorQuantizerPT, self).__init__()
        self.embedding_dim = embedding_dim
        self.n_components = n_components
        self.beta = beta
        self.kmeans = kmeans_loss

        self.codebook = nn.Parameter(
            torch.empty(self.embedding_dim, self.n_components).uniform_(0, 1)
        )
        
        # Store individual loss components for inspection (but not for summing)
        self.last_commitment_loss = None
        self.last_codebook_loss = None
        self.last_kmeans_loss = None
        self.last_vq_loss = None

    def forward(self, x: torch.Tensor, return_losses: bool = True):
        """
        Args:
            x: Input tensor of shape (..., embedding_dim)
               Typically (batch_size, embedding_dim) from encoder
        """
        input_shape = x.shape
        
        losses = {}

        # Flatten to 2D
        flattened = x.reshape(-1, self.embedding_dim)

        # Kmeans loss on flattened 2D tensor
        if self.kmeans and return_losses:
            kmeans_loss_val = deepof.clustering.model_utils_new.compute_kmeans_loss_pt(flattened, self.kmeans)
            losses['kmeans_loss'] = kmeans_loss_val
            self.last_kmeans_loss = kmeans_loss_val.item()

        # Get encodings
        encoding_indices = self.get_code_indices(
            flattened, return_soft_counts=False
        ).long()
        soft_counts = self.get_code_indices(flattened, return_soft_counts=True)

        encodings = F.one_hot(encoding_indices, self.n_components).float()
        quantized = torch.matmul(encodings, self.codebook.T)
        quantized = quantized.reshape(input_shape)

        if return_losses:
            commitment_loss = self.beta * torch.mean(
                (quantized.detach() - x) ** 2
            )
            codebook_loss = torch.mean((quantized - x.detach()) ** 2)
            vq_loss = commitment_loss + codebook_loss
            
            # Store the COMBINED vq_loss in the losses dict (matching TF behavior)
            losses['vq_loss'] = vq_loss
            
            # Store individual components for inspection/logging only
            self.last_commitment_loss = commitment_loss.item()
            self.last_codebook_loss = codebook_loss.item()
            self.last_vq_loss = vq_loss.item()

        if return_losses:
            return quantized, soft_counts, losses
        else:
            return quantized, soft_counts

    def get_code_indices(
        self, flattened_inputs: torch.Tensor, return_soft_counts: bool = False
    ) -> torch.Tensor:
        similarity = torch.matmul(flattened_inputs, self.codebook)
        distances = (
            torch.sum(flattened_inputs**2, dim=1, keepdim=True)
            + torch.sum(self.codebook**2, dim=0)
            - 2 * similarity
        )

        if return_soft_counts:
            similarity = (1 / distances) ** 2
            soft_counts = similarity / torch.sum(similarity, dim=1, keepdim=True)
            return soft_counts

        encoding_indices = torch.argmin(distances, dim=1)
        return encoding_indices

In [10]:
def transfer_vq_weights(tf_model: VectorQuantizer, pt_model: VectorQuantizerPT):
    """Transfers weights from TensorFlow VectorQuantizer to PyTorch version.
    
    Args:
        tf_model: TensorFlow VectorQuantizer model
        pt_model: PyTorch VectorQuantizerPT model
    """
    # Transfer codebook weights
    # TF shape: (embedding_dim, n_components)
    # PT shape: (embedding_dim, n_components)
    # These match, so direct transfer
    codebook_weights = tf_model.codebook.numpy()
    pt_model.codebook.data = torch.from_numpy(codebook_weights)
    
    print(f"✅ Transferred codebook weights: {codebook_weights.shape}")

In [26]:
class TestVectorQuantizerTranslation(unittest.TestCase):
    def setUp(self):
        """Set up the models and transfer weights."""
        tf.keras.backend.clear_session()
        
        self.n_components = 16
        self.embedding_dim = 8
        self.beta = 0.25
        self.kmeans_weight = 0.1
        self.batch_size = 32  # More realistic batch size
        
        # Create TF model
        self.tf_model = VectorQuantizer(
            n_components=self.n_components,
            embedding_dim=self.embedding_dim,
            beta=self.beta,
            kmeans_loss=self.kmeans_weight,
        )
        
        # Create PT model
        self.pt_model = VectorQuantizerPT(
            n_components=self.n_components,
            embedding_dim=self.embedding_dim,
            beta=self.beta,
            kmeans_loss=self.kmeans_weight,
        )
        self.pt_model.eval()
        
        # Initialize TF model with a forward pass (2D input as in real model!)
        dummy_input = tf.random.normal((self.batch_size, self.embedding_dim))
        _ = self.tf_model(dummy_input)
        
        # Transfer weights
        transfer_vq_weights(self.tf_model, self.pt_model)
        
        # Create test data - 2D as encoder outputs!
        np.random.seed(42)
        self.np_input_2d = np.random.randn(
            self.batch_size, self.embedding_dim
        ).astype(np.float32)
        
        # Also create 3D data to test flexibility
        self.np_input_3d = np.random.randn(
            4, 10, self.embedding_dim
        ).astype(np.float32)

    def test_codebook_initialization(self):
        """Test that codebook weights are transferred correctly."""
        tf_codebook = self.tf_model.codebook.numpy()
        pt_codebook = self.pt_model.codebook.detach().cpu().numpy()
        
        np.testing.assert_allclose(tf_codebook, pt_codebook, rtol=1e-6, atol=1e-6)
        print("✅ Codebook initialization test PASSED!")

    def test_code_indices_2d(self):
        """Test code indices with 2D input (main use case)."""
        tf_input = tf.constant(self.np_input_2d)
        tf_flattened = tf.reshape(tf_input, [-1, self.embedding_dim])
        tf_indices = self.tf_model.get_code_indices(
            tf_flattened, return_soft_counts=False
        ).numpy()
        
        pt_input = torch.from_numpy(self.np_input_2d)
        pt_flattened = pt_input.reshape(-1, self.embedding_dim)
        with torch.no_grad():
            pt_indices = self.pt_model.get_code_indices(
                pt_flattened, return_soft_counts=False
            ).cpu().numpy()
        
        np.testing.assert_array_equal(tf_indices, pt_indices)
        print("✅ Code indices 2D test PASSED!")

    def test_soft_counts_2d(self):
        """Test soft counts with 2D input."""
        tf_input = tf.constant(self.np_input_2d)
        tf_flattened = tf.reshape(tf_input, [-1, self.embedding_dim])
        tf_soft = self.tf_model.get_code_indices(
            tf_flattened, return_soft_counts=True
        ).numpy()
        
        pt_input = torch.from_numpy(self.np_input_2d)
        pt_flattened = pt_input.reshape(-1, self.embedding_dim)
        with torch.no_grad():
            pt_soft = self.pt_model.get_code_indices(
                pt_flattened, return_soft_counts=True
            ).cpu().numpy()
        
        np.testing.assert_allclose(tf_soft, pt_soft, rtol=1e-5, atol=1e-6)
        print("✅ Soft counts 2D test PASSED!")

    def test_quantized_output_2d(self):
        """Test quantized output with 2D input."""
        tf_input = tf.constant(self.np_input_2d)
        tf_quantized, tf_soft = self.tf_model(tf_input, training=False)
        
        pt_input = torch.from_numpy(self.np_input_2d)
        with torch.no_grad():
            pt_quantized, pt_soft, _ = self.pt_model(pt_input, return_losses=True)
        
        np.testing.assert_allclose(
            tf_quantized.numpy(), 
            pt_quantized.cpu().numpy(), 
            rtol=1e-5, 
            atol=1e-6
        )
        np.testing.assert_allclose(
            tf_soft.numpy(), 
            pt_soft.cpu().numpy(), 
            rtol=1e-5, 
            atol=1e-6
        )
        print("✅ Quantized output 2D test PASSED!")

    def test_losses_2d(self):
        """Test that losses match between TF and PT with 2D input."""
        tf_input = tf.constant(self.np_input_2d)
        
        with tf.GradientTape():
            tf_quantized, _ = self.tf_model(tf_input, training=True)
            tf_losses = self.tf_model.losses
        
        tf_total_loss = sum(tf_losses).numpy()
        
        pt_input = torch.from_numpy(self.np_input_2d)
        pt_input.requires_grad = True
        
        pt_quantized, pt_soft, pt_losses_dict = self.pt_model(pt_input, return_losses=True)
        
        # Sum all PT losses (now only kmeans_loss and vq_loss)
        pt_total_loss = sum(pt_losses_dict.values()).item()
        
        print(f"\nTF Losses: {[loss.numpy() for loss in tf_losses]}")
        print(f"PT Losses: {[(k, v.item()) for k, v in pt_losses_dict.items()]}")
        print(f"\nTF Total Loss: {tf_total_loss:.6f}")
        print(f"PT Total Loss: {pt_total_loss:.6f}")
        
        # Also print individual components for debugging
        print(f"\nPT Individual Components (for reference):")
        print(f"  Commitment Loss: {self.pt_model.last_commitment_loss:.6f}")
        print(f"  Codebook Loss: {self.pt_model.last_codebook_loss:.6f}")
        
        # Allow slightly larger tolerance for loss comparison due to numerical differences
        np.testing.assert_allclose(tf_total_loss, pt_total_loss, rtol=1e-4, atol=1e-5)
        print("\n✅ Losses 2D test PASSED!")

    def test_full_forward_pass_timing(self):
        """Test full forward pass and compare timing."""
        print("\n" + "="*50)
        print("TIMING COMPARISON (2D Input)")
        print("="*50)
        
        # TF timing
        tf_input = tf.constant(self.np_input_2d)
        tf_start = time.time()
        for _ in range(100):
            tf_quantized, tf_soft = self.tf_model(tf_input, training=False)
        tf_end = time.time()
        tf_time = tf_end - tf_start
        
        # PT timing
        pt_input = torch.from_numpy(self.np_input_2d)
        with torch.no_grad():
            pt_start = time.time()
            for _ in range(100):
                pt_quantized, pt_soft, _ = self.pt_model(pt_input, return_losses=True)
            pt_end = time.time()
        pt_time = pt_end - pt_start
        
        print(f"TensorFlow execution time (100 iters): {tf_time:.4f}s")
        print(f"PyTorch execution time (100 iters): {pt_time:.4f}s")
        print(f"Speedup: {tf_time/pt_time:.5f}x")
        print("="*50)
        
        # Final output comparison
        tf_quantized_final, tf_soft_final = self.tf_model(tf_input, training=False)
        with torch.no_grad():
            pt_quantized_final, pt_soft_final, _ = self.pt_model(pt_input, return_losses=True)
        
        np.testing.assert_allclose(
            tf_quantized_final.numpy(), 
            pt_quantized_final.cpu().numpy(), 
            rtol=1e-5, 
            atol=1e-6
        )
        print("✅ Full forward pass timing test PASSED!")


# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestVectorQuantizerTranslation)
runner.run(suite)

test_code_indices_2d (__main__.TestVectorQuantizerTranslation)
Test code indices with 2D input (main use case). ... ok
test_codebook_initialization (__main__.TestVectorQuantizerTranslation)
Test that codebook weights are transferred correctly. ... ok
test_full_forward_pass_timing (__main__.TestVectorQuantizerTranslation)
Test full forward pass and compare timing. ... 

✅ Transferred codebook weights: (8, 16)
✅ Code indices 2D test PASSED!
✅ Transferred codebook weights: (8, 16)
✅ Codebook initialization test PASSED!
✅ Transferred codebook weights: (8, 16)

TIMING COMPARISON (2D Input)


ERROR
test_losses_2d (__main__.TestVectorQuantizerTranslation)
Test that losses match between TF and PT with 2D input. ... ERROR
test_quantized_output_2d (__main__.TestVectorQuantizerTranslation)
Test quantized output with 2D input. ... ERROR
test_soft_counts_2d (__main__.TestVectorQuantizerTranslation)
Test soft counts with 2D input. ... ok

ERROR: test_full_forward_pass_timing (__main__.TestVectorQuantizerTranslation)
Test full forward pass and compare timing.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_5600\3598182899.py", line 165, in test_full_forward_pass_timing
    pt_quantized, pt_soft, _ = self.pt_model(pt_input, return_losses=True)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\torch\nn\modules\module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "c:\Users\Petron\Desktop\Python_Projects\Deep

✅ Transferred codebook weights: (8, 16)
✅ Transferred codebook weights: (8, 16)
✅ Transferred codebook weights: (8, 16)
✅ Soft counts 2D test PASSED!


<unittest.runner.TextTestResult run=6 errors=3 failures=0>

# Full VQVAE

In [11]:
# Cell 1: PyTorch VQVAE Model (Corrected for actual input dimensions)

import deepof.clustering.models_new


class VQVAEPT(nn.Module):
    """
    PyTorch implementation of the VQ-VAE model adapted to the DeepOF setting.
    
    Note: This version handles the actual DeepOF input format where:
    - x: (B, T, node_features) - flattened node features
    - a: (B, T, edge_features) - flattened edge features
    """
    
    def __init__(
        self,
        input_shape: tuple,
        edge_feature_shape: tuple,
        adjacency_matrix: np.ndarray,
        latent_dim: int,
        n_components: int,
        beta: float = 1.0,
        kmeans_loss: float = 0.0,
        use_gnn: bool = True,
        encoder_type: str = "recurrent",
        interaction_regularization: float = 0.0,
    ):
        """Initialize a VQ-VAE model.

        Args:
            input_shape (tuple): Shape of the input (time_steps, node_features).
            edge_feature_shape (tuple): Shape of edge features (time_steps, edge_features).
            adjacency_matrix (np.ndarray): Adjacency matrix for GNN.
            latent_dim (int): Dimensionality of the latent space.
            n_components (int): Number of embeddings (clusters) in the codebook.
            beta (float): Beta parameter of the VQ loss.
            kmeans_loss (float): Regularization parameter for the Gram matrix.
            use_gnn (bool): Whether to use GNN in encoder.
            encoder_type (str): Type of encoder ("recurrent", "TCN", or "transformer").
            interaction_regularization (float): Regularization parameter for interactions.
        """
        super().__init__()
        
        time_steps, n_nodes, n_features_per_node = input_shape
        self.input_n_nodes = n_nodes
        self.input_n_features_per_node = n_features_per_node
        self.window_size = time_steps
        self.latent_dim = latent_dim
        self.n_components = n_components
        self.encoder_type = encoder_type
        
        # Initialize encoder based on type
        if encoder_type == "recurrent":
            self.encoder = deepof.clustering.models_new.RecurrentEncoderPT(
                input_shape=input_shape,
                edge_feature_shape=edge_feature_shape,
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
                interaction_regularization=interaction_regularization,
            )
            decoder_output_features = n_nodes * n_features_per_node
            self.decoder = deepof.clustering.models_new.RecurrentDecoderPT(
                output_shape=(time_steps, decoder_output_features),
                latent_dim=latent_dim,
            )
        elif encoder_type == "TCN":
            self.encoder = deepof.clustering.models_new.TCNEncoderPT(
                input_shape=input_shape,
                edge_feature_shape=edge_feature_shape,
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
                interaction_regularization=interaction_regularization,
            )
            decoder_output_features = n_nodes * n_features_per_node
            self.decoder = deepof.clustering.models_new.TCNDecoderPT(
                output_shape=(time_steps, decoder_output_features),
                latent_dim=latent_dim,
            ) 
        elif encoder_type == "transformer":
            self.encoder = deepof.clustering.models_new.TFMEncoderPT(
                input_shape=input_shape,
                edge_feature_shape=edge_feature_shape,
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
            )
            decoder_output_features = n_nodes * n_features_per_node
            self.decoder = deepof.clustering.models_new.TFMDecoderPT(
                output_shape=(time_steps, decoder_output_features),
                latent_dim=latent_dim,
                num_layers=2,
                num_heads=8,
                dff=128,
                dropout_rate=0.2,
                teacher_forcing_mode="zeros",
                input_dropout_p=0.5,
                self_attn_diag_only=False,
            )
        else:
            raise ValueError(f"Unknown encoder_type: {encoder_type}")
        
        # Initialize Vector Quantizer
        self.vq_layer = VectorQuantizerPT(
            n_components=n_components,
            embedding_dim=latent_dim,
            beta=beta,
            kmeans_loss=kmeans_loss,
        )

    def forward(
        self,
        x: torch.Tensor,
        a: torch.Tensor,
        return_losses: bool = True,
        return_all_outputs: bool = False,
    ):
        """
        Forward pass through the VQ-VAE model.
        
        Args:
            x: Input node features (B, T, N, F)
            a: Input edge features (B, T, E, F_edge)
            return_losses: Whether to compute and return VQ losses
            return_all_outputs: Whether to return all intermediate outputs
        """
        # Encode
        encoder_output = self.encoder(x, a)  # Shape: (B, latent_dim)
        
        # Vector Quantization
        if return_losses:
            quantized_latents, soft_counts, vq_losses = self.vq_layer(
                encoder_output, return_losses=True
            )
        else:
            quantized_latents, soft_counts = self.vq_layer(
                encoder_output, return_losses=False
            )
            vq_losses = {}
        
        # Prepare input for decoder (flatten spatial dimensions for teacher forcing)
        B, T, N, F = x.shape
        x_for_decoder = x.view(B, T, N * F)  # Flatten to (B, T, node_features)
        
        # Decode from QUANTIZED latents (main path)
        encoding_reconstruction_dist = self.decoder(quantized_latents, x_for_decoder)
        
        # Decode from ORIGINAL encoder output (bypass path for gradient flow)
        reconstruction_dist = self.decoder(encoder_output, x_for_decoder)
        
        # Handle transformer decoder outputs (which return attention weights)
        if self.encoder_type == "transformer":
            if isinstance(encoding_reconstruction_dist, tuple):
                encoding_reconstruction_dist = encoding_reconstruction_dist[0]
            if isinstance(reconstruction_dist, tuple):
                reconstruction_dist = reconstruction_dist[0]
        
        if return_all_outputs:
            return (
                encoding_reconstruction_dist,
                reconstruction_dist,
                quantized_latents,
                soft_counts,
                encoder_output,
                vq_losses if return_losses else None,
            )
        else:
            if return_losses:
                return encoding_reconstruction_dist, reconstruction_dist, vq_losses
            else:
                return encoding_reconstruction_dist, reconstruction_dist

    @torch.no_grad()
    def encode(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Inference-only: Get encoder output. Equivalent to TF 'encoder' model."""
        return self.encoder(x, a)

    @torch.no_grad()
    def group(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Inference-only: Get quantized latents. Equivalent to TF 'grouper' model."""
        encoder_output = self.encoder(x, a)
        quantized, _ = self.vq_layer(encoder_output, return_losses=False)
        return quantized

    @torch.no_grad()
    def soft_group(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Inference-only: Get soft cluster assignments. Equivalent to TF 'soft_grouper' model."""
        encoder_output = self.encoder(x, a)
        _, soft_counts = self.vq_layer(encoder_output, return_losses=False)
        return soft_counts

    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Full reconstruction from input through VQ-VAE."""
        encoding_recon_dist, _ = self.forward(x, a, return_losses=False)
        return encoding_recon_dist.mean
    
    def get_codebook_usage(self, data_loader, max_samples: int = 10000):
        """Compute codebook usage statistics over a dataset."""
        self.eval()
        all_indices = []
        all_soft_counts = []
        samples_seen = 0
        
        with torch.no_grad():
            for x, a, *_ in data_loader:
                x = x.to(next(self.parameters()).device)
                a = a.to(next(self.parameters()).device)
                
                encoder_output = self.encoder(x, a)
                flattened = encoder_output.reshape(-1, self.latent_dim)
                
                indices = self.vq_layer.get_code_indices(
                    flattened, return_soft_counts=False
                )
                soft_counts = self.vq_layer.get_code_indices(
                    flattened, return_soft_counts=True
                )
                
                all_indices.append(indices.cpu())
                all_soft_counts.append(soft_counts.cpu())
                
                samples_seen += x.size(0)
                if samples_seen >= max_samples:
                    break
        
        all_indices = torch.cat(all_indices)
        all_soft_counts = torch.cat(all_soft_counts)
        
        usage_counts = torch.bincount(all_indices, minlength=self.n_components)
        
        # Compute perplexity
        avg_probs = all_soft_counts.mean(dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        return usage_counts, perplexity.item()

In [13]:
def train_vqvae_epoch(
    model: VQVAEPT,
    train_loader,
    optimizer,
    device,
    epoch: int,
):
    """
    Training loop for one epoch of VQ-VAE.
    
    Returns:
        Dictionary of average losses for the epoch
    """
    model.train()
    
    total_loss_sum = 0.0
    encoding_recon_loss_sum = 0.0
    recon_loss_sum = 0.0
    vq_loss_sum = 0.0
    kmeans_loss_sum = 0.0
    n_batches = 0
    
    for batch_idx, (x, a, y) in enumerate(train_loader):
        x, a, y = x.to(device), a.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        encoding_recon_dist, recon_dist, vq_losses = model(
            x, a, return_losses=True, return_all_outputs=False
        )
        
        # Compute reconstruction losses (negative log-likelihood)
        encoding_recon_loss = -encoding_recon_dist.log_prob(y).mean()
        recon_loss = -recon_dist.log_prob(y).mean()
        
        # Get VQ losses
        vq_loss = vq_losses['vq_loss']
        kmeans_loss = vq_losses.get('kmeans_loss', torch.tensor(0.0).to(device))
        
        # Total loss (matching TF implementation)
        total_loss = encoding_recon_loss + recon_loss + vq_loss + kmeans_loss
        
        # Backward pass
        total_loss.backward()
        optimizer.step()
        
        # Accumulate losses
        total_loss_sum += total_loss.item()
        encoding_recon_loss_sum += encoding_recon_loss.item()
        recon_loss_sum += recon_loss.item()
        vq_loss_sum += vq_loss.item()
        kmeans_loss_sum += kmeans_loss.item()
        n_batches += 1
    
    # Compute cluster population
    with torch.no_grad():
        x_sample, a_sample, _ = next(iter(train_loader))
        x_sample, a_sample = x_sample.to(device), a_sample.to(device)
        soft_counts = model.soft_group(x_sample, a_sample)
        cluster_assignments = torch.argmax(soft_counts, dim=1)
        n_populated = len(torch.unique(cluster_assignments))
    
    return {
        'total_loss': total_loss_sum / n_batches,
        'encoding_reconstruction_loss': encoding_recon_loss_sum / n_batches,
        'reconstruction_loss': recon_loss_sum / n_batches,
        'vq_loss': vq_loss_sum / n_batches,
        'kmeans_loss': kmeans_loss_sum / n_batches,
        'n_populated_clusters': n_populated,
    }


@torch.no_grad()
def validate_vqvae_epoch(
    model: VQVAEPT,
    val_loader,
    device,
):
    """
    Validation loop for one epoch of VQ-VAE.
    
    Returns:
        Dictionary of average losses for the epoch
    """
    model.eval()
    
    total_loss_sum = 0.0
    encoding_recon_loss_sum = 0.0
    recon_loss_sum = 0.0
    vq_loss_sum = 0.0
    kmeans_loss_sum = 0.0
    n_batches = 0
    
    for x, a, y in val_loader:
        x, a, y = x.to(device), a.to(device), y.to(device)
        
        # Forward pass
        encoding_recon_dist, recon_dist, vq_losses = model(
            x, a, return_losses=True, return_all_outputs=False
        )
        
        # Compute reconstruction losses
        encoding_recon_loss = -encoding_recon_dist.log_prob(y).mean()
        recon_loss = -recon_dist.log_prob(y).mean()
        
        # Get VQ losses
        vq_loss = vq_losses['vq_loss']
        kmeans_loss = vq_losses.get('kmeans_loss', torch.tensor(0.0).to(device))
        
        # Total loss
        total_loss = encoding_recon_loss + recon_loss + vq_loss + kmeans_loss
        
        # Accumulate losses
        total_loss_sum += total_loss.item()
        encoding_recon_loss_sum += encoding_recon_loss.item()
        recon_loss_sum += recon_loss.item()
        vq_loss_sum += vq_loss.item()
        kmeans_loss_sum += kmeans_loss.item()
        n_batches += 1
    
    # Compute cluster population
    x_sample, a_sample, _ = next(iter(val_loader))
    x_sample, a_sample = x_sample.to(device), a_sample.to(device)
    soft_counts = model.soft_group(x_sample, a_sample)
    cluster_assignments = torch.argmax(soft_counts, dim=1)
    n_populated = len(torch.unique(cluster_assignments))
    
    return {
        'val_total_loss': total_loss_sum / n_batches,
        'val_encoding_reconstruction_loss': encoding_recon_loss_sum / n_batches,
        'val_reconstruction_loss': recon_loss_sum / n_batches,
        'val_vq_loss': vq_loss_sum / n_batches,
        'val_kmeans_loss': kmeans_loss_sum / n_batches,
        'val_n_populated_clusters': n_populated,
    }


def train_vqvae_full(
    model: VQVAEPT,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs: int,
    log_interval: int = 10,
):
    """
    Complete training loop for VQ-VAE.
    
    Args:
        model: The VQVAEPT model
        train_loader: Training data loader
        val_loader: Validation data loader
        optimizer: PyTorch optimizer
        device: Device to train on
        num_epochs: Number of epochs to train
        log_interval: How often to print progress
        
    Returns:
        Dictionary containing training history
    """
    history = {
        'train': [],
        'val': []
    }
    
    for epoch in range(num_epochs):
        # Train
        train_metrics = train_vqvae_epoch(model, train_loader, optimizer, device, epoch)
        history['train'].append(train_metrics)
        
        # Validate
        val_metrics = validate_vqvae_epoch(model, val_loader, device)
        history['val'].append(val_metrics)
        
        # Log progress
        if (epoch + 1) % log_interval == 0 or epoch == 0:
            print(f"\nEpoch {epoch + 1}/{num_epochs}")
            print(f"  Train - Total: {train_metrics['total_loss']:.4f}, "
                  f"Enc Recon: {train_metrics['encoding_reconstruction_loss']:.4f}, "
                  f"Recon: {train_metrics['reconstruction_loss']:.4f}, "
                  f"VQ: {train_metrics['vq_loss']:.4f}, "
                  f"KMeans: {train_metrics['kmeans_loss']:.4f}, "
                  f"Clusters: {train_metrics['n_populated_clusters']}")
            print(f"  Val   - Total: {val_metrics['val_total_loss']:.4f}, "
                  f"Enc Recon: {val_metrics['val_encoding_reconstruction_loss']:.4f}, "
                  f"Recon: {val_metrics['val_reconstruction_loss']:.4f}, "
                  f"VQ: {val_metrics['val_vq_loss']:.4f}, "
                  f"KMeans: {val_metrics['val_kmeans_loss']:.4f}, "
                  f"Clusters: {val_metrics['val_n_populated_clusters']}")
    
    return history

In [ ]:
def transfer_vqvae_weights(tf_vqvae, pt_vqvae):
    """
    Transfer weights from TensorFlow VQVAE to PyTorch VQVAEPT.
    
    Args:
        tf_vqvae: TensorFlow VQVAE instance
        pt_vqvae: PyTorch VQVAEPT instance
    """
    print("Transferring VQ-VAE weights...")
    
    # Transfer encoder weights (assumed already done via encoder-specific transfer functions)
    # transfer_encoder_weights(tf_vqvae.encoder, pt_vqvae.encoder)
    print("  ⚠️  Encoder weights - use encoder-specific transfer function")
    
    # Transfer VQ layer weights
    tf_vq_layer = tf_vqvae.vqvae.get_layer('vector_quantizer')
    transfer_vq_weights(tf_vq_layer, pt_vqvae.vq_layer)
    print("  ✅ VQ layer weights transferred")
    
    # Transfer decoder weights (assumed already done via decoder-specific transfer functions)
    # transfer_decoder_weights(tf_vqvae.decoder, pt_vqvae.decoder)
    print("  ⚠️  Decoder weights - use decoder-specific transfer function")
    
    print("✅ VQ-VAE weight transfer complete!")

In [52]:
def _flatten_keras_tensors(x):
    if isinstance(x, (list, tuple)):
        out = []
        for xi in x:
            out.extend(_flatten_keras_tensors(xi))
        return out
    return [x]

def _layer_from_tensor(t):
    kh = getattr(t, "_keras_history", None)
    return kh[0] if isinstance(kh, tuple) else getattr(kh, "layer", None)

def _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch):
    # Find CensNetConv and its inbound tensors
    gnn_tf = next(l for l in tf_enc.layers if l.__class__.__name__ == "CensNetConv")
    node = gnn_tf._inbound_nodes[-1]  # use last call
    in_ts = getattr(node, "input_tensors", None)
    if in_ts is None:
        in_ts = getattr(node, "inputs")
    in_ts = _flatten_keras_tensors(in_ts)

    # Among inbound tensors, pick the two with rank 3 and last dim == 2*latent
    cand = [t for t in in_ts if len(t.shape) == 3]
    # Disambiguate by producer: nested Functional input feature dim
    pairs = []
    for t in cand:
        prod = _layer_from_tensor(t)   # nested Functional (recurrent block)
        if isinstance(prod, tf.keras.Model):
            in_last = int(prod.inputs[0].shape[-1])
            pairs.append((t, prod, in_last))

    # Map by input feature dim
    tf_node_pre_t = next(t for t, prod, in_last in pairs if in_last == node_in_ch)
    tf_edge_pre_t = next(t for t, prod, in_last in pairs if in_last == edge_in_ch)
    tf_node_block = _layer_from_tensor(tf_node_pre_t)
    tf_edge_block = _layer_from_tensor(tf_edge_pre_t)
    return gnn_tf, tf_node_pre_t, tf_edge_pre_t, tf_node_block, tf_edge_block

In [7]:
import unittest
import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
import time
import sys
from io import StringIO


class TestVQVAETranslation(unittest.TestCase):
    
    @classmethod
    def setUpClass(cls):
        """Set up once for all tests."""
        print("\n" + "="*70)
        print(" VQ-VAE TRANSLATION TEST SUITE")
        print("="*70)
    
    def setUp(self):
        """Set up models for testing."""
        tf.keras.backend.clear_session()
        
        # Test parameters
        self.batch_size = 64
        self.time_steps = 25
        self.num_nodes = 11
        self.num_edges = 11
        self.n_node_features = 3
        self.n_edge_features = 1
        self.latent_dim = 6
        self.n_components = 10
        self.beta = 0.25
        self.kmeans_loss = 0.1
        

        # TensorFlow expects (batch, time, total_features)
        self.input_shape_tf = (self.batch_size, self.time_steps, self.num_nodes * self.n_node_features)
        self.edge_feature_shape_tf = (self.batch_size, self.time_steps, self.num_edges * self.n_edge_features)
        
        # PyTorch expects (time, nodes, features_per_node) for input_shape parameter
        self.input_shape_pt = (self.time_steps, self.num_nodes, self.n_node_features)
        self.edge_feature_shape_pt = (self.time_steps, self.num_edges, self.n_edge_features)
        
        # Create dummy adjacency matrix
        m = np.zeros((self.num_nodes, self.num_nodes))
        ui = np.triu_indices(self.num_nodes)
        num_possible_edges = len(ui[0])
        c = np.random.choice(num_possible_edges, min(self.num_edges, num_possible_edges), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        m += m.T # Make symmetric
        self.adjacency_matrix = m

        use_gnn=True
        
        # Create TensorFlow model
        self.tf_model = deepof.models.VQVAE(
            input_shape=self.input_shape_tf,
            edge_feature_shape=self.edge_feature_shape_tf,
            adjacency_matrix=self.adjacency_matrix,
            latent_dim=self.latent_dim,
            n_components=self.n_components,
            beta=self.beta,
            kmeans_loss=self.kmeans_loss,
            use_gnn=use_gnn,  # Simplified for testing
            encoder_type="recurrent",
        )
        
        # Create PyTorch model  
        self.pt_model = deepof.clustering.models_new.VQVAEPT(
            input_shape=self.input_shape_pt,  # Without batch dimension
            edge_feature_shape=self.edge_feature_shape_pt,
            adjacency_matrix=self.adjacency_matrix,
            latent_dim=self.latent_dim,
            n_components=self.n_components,
            beta=self.beta,
            kmeans_loss=self.kmeans_loss,
            use_gnn=use_gnn,
            encoder_type="recurrent",
        )
        self.pt_model.eval()
        
        # Create test data
        np.random.seed(42)
        
        # Generate TensorFlow format data (B, T, total_features)
        self.x_test_tf = np.random.randn(*self.input_shape_tf).astype(np.float32)
        self.a_test_tf = np.random.randn(*self.edge_feature_shape_tf).astype(np.float32)
        self.y_test_tf = np.random.randn(*self.input_shape_tf).astype(np.float32)
        
        # Reshape to PyTorch format (B, T, nodes, features_per_node)
        self.x_test_pt = self.x_test_tf.reshape(
            self.batch_size, self.time_steps, self.num_nodes, self.n_node_features
        )
        self.a_test_pt = self.a_test_tf.reshape(
            self.batch_size, self.time_steps, self.num_edges, self.n_edge_features
        )
        self.y_test_pt = self.y_test_tf.reshape(
            self.batch_size, self.time_steps, self.num_nodes, self.n_node_features
        )
        
        # Initialize TF model with forward pass
        _ = self.tf_model([
            tf.constant(self.x_test_tf),
            tf.constant(self.a_test_tf)
        ], training=False)
        
        # Suppress transfer weight output
        old_stdout = sys.stdout
        sys.stdout = StringIO()
        
        # Transfer VQ weights only (encoder/decoder handled separately)
        transfer_vq_weights(
            self.tf_model.vqvae.get_layer('vector_quantizer'),
            self.pt_model.vq_layer
        )
        
        sys.stdout = old_stdout

    def test_1_model_structure(self):
        """Test that model components exist and have correct structure."""
        # Check TF model
        self.assertIsNotNone(self.tf_model.encoder)
        self.assertIsNotNone(self.tf_model.decoder)
        self.assertIsNotNone(self.tf_model.grouper)
        self.assertIsNotNone(self.tf_model.soft_grouper)
        
        # Check PT model
        self.assertIsNotNone(self.pt_model.encoder)
        self.assertIsNotNone(self.pt_model.decoder)
        self.assertIsNotNone(self.pt_model.vq_layer)

    def test_2_encoder_output_shape(self):
        """Test encoder output shapes match."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_encoder_out = self.tf_model.encoder([tf_input, tf_a])
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_encoder_out = self.pt_model.encoder(pt_input, pt_a)
        
        self.assertEqual(
            tuple(tf_encoder_out.shape),
            tuple(pt_encoder_out.shape),
            "Encoder output shapes don't match"
        )

    def test_3_vq_layer_quantization(self):
        """Test VQ layer produces same quantized outputs."""
        # Create dummy encoder output
        encoder_out = np.random.randn(self.batch_size, self.latent_dim).astype(np.float32)
        
        # TF quantization
        tf_enc = tf.constant(encoder_out)
        tf_quantized, tf_soft = self.tf_model.vqvae.get_layer('vector_quantizer')(tf_enc)
        
        # PT quantization
        pt_enc = torch.from_numpy(encoder_out)
        with torch.no_grad():
            pt_quantized, pt_soft = self.pt_model.vq_layer(pt_enc, return_losses=False)
        
        # Compare quantized outputs
        np.testing.assert_allclose(
            tf_quantized.numpy(),
            pt_quantized.cpu().numpy(),
            rtol=1e-5,
            atol=1e-6
        )
        
        # Compare soft counts
        np.testing.assert_allclose(
            tf_soft.numpy(),
            pt_soft.cpu().numpy(),
            rtol=1e-5,
            atol=1e-6
        )

    def test_4_grouper_outputs(self):
        """Test grouper (quantized latents) outputs match."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_grouped = self.tf_model.grouper([tf_input, tf_a])
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_grouped = self.pt_model.group(pt_input, pt_a)
        
        self.assertEqual(
            tuple(tf_grouped.shape),
            tuple(pt_grouped.shape),
            "Grouper output shapes don't match"
        )

    def test_5_soft_grouper_outputs(self):
        """Test soft_grouper (soft counts) outputs match."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_soft = self.tf_model.soft_grouper([tf_input, tf_a])
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_soft = self.pt_model.soft_group(pt_input, pt_a)
        
        self.assertEqual(
            tuple(tf_soft.shape),
            tuple(pt_soft.shape),
            "Soft grouper output shapes don't match"
        )
        
        pt_soft_sums = pt_soft.sum(dim=1)
        np.testing.assert_allclose(
            pt_soft_sums.cpu().numpy(),
            np.ones(self.batch_size),
            rtol=1e-5,
            atol=1e-6,
            err_msg="Soft counts don't sum to 1"
        )

    def test_6_forward_pass_outputs(self):
        """Test full forward pass output shapes."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_output = self.tf_model.vqvae([tf_input, tf_a], training=False)
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_enc_recon, pt_recon = self.pt_model(pt_input, pt_a, return_losses=False)
        
        # TF output shape: (B, T, total_features) - flattened
        expected_shape_tf = (self.batch_size, self.time_steps, 
                            self.num_nodes * self.n_node_features)
        
        # PT output shape: (B, T, total_features) - also flattened (matches decoder output)
        expected_shape_pt = (self.batch_size, self.time_steps, 
                            self.num_nodes * self.n_node_features)
        
        self.assertEqual(
            tuple(tf_output.mean().shape),
            expected_shape_tf,
            "TF output shape doesn't match expected"
        )
        
        self.assertEqual(
            tuple(pt_enc_recon.mean.shape),
            expected_shape_pt,
            "PT encoding reconstruction shape doesn't match expected"
        )
        
        self.assertEqual(
            tuple(pt_recon.mean.shape),
            expected_shape_pt,
            "PT reconstruction shape doesn't match expected"
        )

    def test_7_loss_computation(self):
        """Test that losses can be computed properly."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        pt_y = torch.from_numpy(self.y_test_pt)
        
        # Flatten y to match decoder output format
        B, T, N, F = pt_y.shape
        pt_y_flat = pt_y.view(B, T, N * F)
        
        self.pt_model.train()
        
        enc_recon_dist, recon_dist, vq_losses = self.pt_model(
            pt_input, pt_a, return_losses=True
        )
        
        # Compute reconstruction losses with flattened ground truth
        enc_recon_loss = -enc_recon_dist.log_prob(pt_y_flat).mean()
        recon_loss = -recon_dist.log_prob(pt_y_flat).mean()
        
        self.assertIn('vq_loss', vq_losses)
        if self.kmeans_loss > 0:
            self.assertIn('kmeans_loss', vq_losses)
        
        self.assertTrue(torch.isfinite(enc_recon_loss).all())
        self.assertTrue(torch.isfinite(recon_loss).all())
        self.assertTrue(torch.isfinite(vq_losses['vq_loss']).all())

    def test_8_backward_pass(self):
        """Test that gradients flow properly through the model."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        pt_y = torch.from_numpy(self.y_test_pt)
        
        # Flatten y to match decoder output format
        B, T, N, F = pt_y.shape
        pt_y_flat = pt_y.view(B, T, N * F)
        
        pt_input.requires_grad = True
        
        self.pt_model.train()
        
        enc_recon_dist, recon_dist, vq_losses = self.pt_model(
            pt_input, pt_a, return_losses=True
        )
        
        # Compute total loss with flattened ground truth
        enc_recon_loss = -enc_recon_dist.log_prob(pt_y_flat).mean()
        recon_loss = -recon_dist.log_prob(pt_y_flat).mean()
        total_loss = enc_recon_loss + recon_loss + sum(vq_losses.values())
        
        total_loss.backward()
        
        self.assertIsNotNone(self.pt_model.vq_layer.codebook.grad)
        self.assertTrue(torch.isfinite(self.pt_model.vq_layer.codebook.grad).all())

    def test_90_cluster_population(self):
        """Test cluster population computation."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        with torch.no_grad():
            soft_counts = self.pt_model.soft_group(pt_input, pt_a)
            cluster_assignments = torch.argmax(soft_counts, dim=1)
            unique_clusters = torch.unique(cluster_assignments)
            n_populated = len(unique_clusters)
        
        self.assertGreater(n_populated, 0)
        self.assertLessEqual(n_populated, self.n_components)

    def test_91_inference_methods(self):
        """Test all inference methods work correctly."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        self.pt_model.eval()
        
        with torch.no_grad():
            # Test encode
            encoded = self.pt_model.encode(pt_input, pt_a)
            self.assertEqual(encoded.shape, (self.batch_size, self.latent_dim))
            
            # Test group
            quantized = self.pt_model.group(pt_input, pt_a)
            self.assertEqual(quantized.shape, (self.batch_size, self.latent_dim))
            
            # Test soft_group
            soft_counts = self.pt_model.soft_group(pt_input, pt_a)
            self.assertEqual(soft_counts.shape, (self.batch_size, self.n_components))
            
            # Test reconstruct - output is FLATTENED
            reconstruction = self.pt_model.reconstruct(pt_input, pt_a)
            expected_shape = (self.batch_size, self.time_steps, 
                            self.num_nodes * self.n_node_features)
            self.assertEqual(tuple(reconstruction.shape), expected_shape)

    def test_92_return_all_outputs(self):
        """Test return_all_outputs flag."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        with torch.no_grad():
            outputs = self.pt_model(
                pt_input, pt_a, 
                return_losses=True, 
                return_all_outputs=True
            )
        
        self.assertEqual(len(outputs), 6)
        
        (enc_recon_dist, recon_dist, quantized, soft_counts, 
        encoder_output, vq_losses) = outputs
        
        self.assertEqual(quantized.shape, (self.batch_size, self.latent_dim))
        self.assertEqual(soft_counts.shape, (self.batch_size, self.n_components))
        self.assertEqual(encoder_output.shape, (self.batch_size, self.latent_dim))
        self.assertIsInstance(vq_losses, dict)

    def test_93_timing_comparison(self):
        """Compare inference speed between TF and PT."""
        print("\n" + "="*70)
        print(" TIMING COMPARISON")
        print("="*70)
        
        n_iters = 50
        
        # TensorFlow timing
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        
        tf_start = time.time()
        for _ in range(n_iters):
            _ = self.tf_model.vqvae([tf_input, tf_a], training=False)
        tf_time = time.time() - tf_start
        
        # PyTorch timing
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        with torch.no_grad():
            pt_start = time.time()
            for _ in range(n_iters):
                _ = self.pt_model(pt_input, pt_a, return_losses=False)
            pt_time = time.time() - pt_start
        
        print(f"TensorFlow: {tf_time:.4f}s ({tf_time/n_iters*1000:.2f}ms per iter)")
        print(f"PyTorch:    {pt_time:.4f}s ({pt_time/n_iters*1000:.2f}ms per iter)")
        print(f"Speedup:    {tf_time/pt_time:.2f}x")
        print("="*70)
        
        self.assertGreater(tf_time, 0)
        self.assertGreater(pt_time, 0)


    def test_encoder_nested_blocks_direct_parity(self):
        print("\n" + "="*70)
        print(" TEST: Encoder nested blocks direct parity")
        print("="*70)
        assert self.pt_model.encoder.use_gnn

        # 1) Transfer weights (uses GNN inbound mapping)
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)

        # 2) Locate the exact nested blocks feeding CensNetConv
        tf_enc = self.tf_model.encoder.get_layer("recurrent_encoder")
        gnn_tf = next(l for l in tf_enc.layers if l.__class__.__name__ == "CensNetConv")
        node = gnn_tf._inbound_nodes[0]
        in_tensors = getattr(node, "input_tensors", None)
        if in_tensors is None:
            in_tensors = getattr(node, "inputs")
        in_tensors = _flatten_keras_tensors(in_tensors)
        tf_node_pre_t = in_tensors[0]
        tf_edge_pre_t = in_tensors[-1]
        tf_node_block = _layer_from_tensor(tf_node_pre_t)
        tf_edge_block = _layer_from_tensor(tf_edge_pre_t)

        # 3) Build direct-call models for the nested blocks
        #    Note: calling the nested Model on a new Input is fine.
        B, T = self.batch_size, self.time_steps
        N, E = self.num_nodes, self.num_edges
        Fn, Fe = self.n_node_features, self.n_edge_features

        tf_node_inp = tf.keras.Input(shape=(N, T, Fn))
        tf_edge_inp = tf.keras.Input(shape=(E, T, Fe))
        tf_node_direct = tf.keras.Model(tf_node_inp, tf_node_block(tf_node_inp), name="tf_node_direct")
        tf_edge_direct = tf.keras.Model(tf_edge_inp, tf_edge_block(tf_edge_inp), name="tf_edge_direct")

        # 4) Prepare identical inputs (same as you used in the reshape test)
        tf_nodes_in_np = self.x_test_tf.reshape(B, T, N, Fn).transpose(0, 2, 1, 3)
        tf_edges_in_np = self.a_test_tf.reshape(B, T, E, Fe).transpose(0, 2, 1, 3)

        # 5) Forward both TF and PT blocks on the same inputs
        tf_node_pre = tf_node_direct(tf_nodes_in_np, training=False).numpy()
        tf_edge_pre = tf_edge_direct(tf_edges_in_np, training=False).numpy()

        self.pt_model.encoder.eval()
        with torch.no_grad():
            pt_nodes_in = torch.from_numpy(tf_nodes_in_np)
            pt_edges_in = torch.from_numpy(tf_edges_in_np)
            pt_node_pre = self.pt_model.encoder.node_recurrent_block(pt_nodes_in).cpu().numpy()
            pt_edge_pre = self.pt_model.encoder.edge_recurrent_block(pt_edges_in).cpu().numpy()

        # 6) Compare
        def rep(name, a, b, rtol=1e-5, atol=1e-6):
            d = np.abs(a - b)
            print(f"{name}: max {d.max():.8f} | mean {d.mean():.8f}")
            np.testing.assert_allclose(a, b, rtol=rtol, atol=atol, err_msg=f"{name} mismatch")

        rep("Node block output", tf_node_pre, pt_node_pre)
        rep("Edge block output", tf_edge_pre, pt_edge_pre)

        print("✅ Encoder nested blocks direct parity passed")


    def test_000gnn_reshape_and_intermediate_parity(self):
        """
        Compare, step by step, what TF vs PT feed into and get out of the GNN path:
        1) Inputs to the node/edge recurrent blocks (after TF reshapes)
        2) Outputs of the node/edge recurrent blocks (pre-GNN embeddings)
        3) Outputs of the GNN (nodes and edges)
        4) Flatten+concat tensor fed to final Dense
        """
        print("\n" + "="*70)
        print(" TEST: GNN RESHAPE + INTERMEDIATE PARITY")
        print("="*70)
       
        assert self.pt_model.encoder.use_gnn, "This test is for use_gnn=True"

        # 0) Transfer weights
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)

        tf_enc = self.tf_model.encoder.get_layer("recurrent_encoder")
        node_in_ch = int(self.pt_model.encoder.node_recurrent_block.conv1d.in_channels)
        edge_in_ch = int(self.pt_model.encoder.edge_recurrent_block.conv1d.in_channels)

        gnn_tf, tf_node_pre_t, tf_edge_pre_t, tf_node_block, tf_edge_block = \
            _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch)

        tf_tap = tf.keras.Model(
            inputs=tf_enc.inputs,
            outputs=[tf_node_pre_t, tf_edge_pre_t, gnn_tf.output]
        )

        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_node_pre, tf_edge_pre, tf_gnn_out = tf_tap([tf_input, tf_a], training=False)
        tf_node_pre = tf_node_pre.numpy()
        tf_edge_pre = tf_edge_pre.numpy()
        tf_gnn_nodes = tf_gnn_out[0].numpy()
        tf_gnn_edges = tf_gnn_out[1].numpy()

        # Sanity: the inputs to the recurrent blocks match
        B, T = self.batch_size, self.time_steps
        N, E = self.num_nodes, self.num_edges
        Fn, Fe = self.n_node_features, self.n_edge_features
        tf_nodes_in_np = self.x_test_tf.reshape(B, T, N, Fn).transpose(0, 2, 1, 3)
        tf_edges_in_np = self.a_test_tf.reshape(B, T, E, Fe).transpose(0, 2, 1, 3)

        pt_enc = self.pt_model.encoder
        pt_enc.eval()
        with torch.no_grad():
            pt_nodes_in = torch.from_numpy(tf_nodes_in_np)
            pt_edges_in = torch.from_numpy(tf_edges_in_np)
            node_pre = pt_enc.node_recurrent_block(pt_nodes_in)  # (B, N, 2*ld)
            edge_pre = pt_enc.edge_recurrent_block(pt_edges_in)  # (B, E, 2*ld)
            adj_tuple = (pt_enc.laplacian, pt_enc.edge_laplacian, pt_enc.incidence)
            pt_gnn_nodes, pt_gnn_edges = pt_enc.spatial_gnn_block([node_pre, adj_tuple, edge_pre])

        pt_node_pre_np = node_pre.cpu().numpy()
        pt_edge_pre_np = edge_pre.cpu().numpy()
        pt_gnn_nodes_np = pt_gnn_nodes.cpu().numpy()
        pt_gnn_edges_np = pt_gnn_edges.cpu().numpy()

        def rep(name, a, b, rtol=1e-5, atol=1e-6):
            d = np.abs(a - b)
            print(f"{name}: shape {a.shape} | max {d.max():.8f} | mean {d.mean():.8f}")
            np.testing.assert_allclose(a, b, rtol=rtol, atol=atol, err_msg=f"{name} mismatch")

        print("\nComparing pre-GNN embeddings (recurrent outputs)")
        rep("Nodes pre-GNN", tf_node_pre, pt_node_pre_np)
        rep("Edges pre-GNN", tf_edge_pre, pt_edge_pre_np)

        print("\nComparing GNN outputs")
        rep("GNN nodes", tf_gnn_nodes, pt_gnn_nodes_np)
        rep("GNN edges", tf_gnn_edges, pt_gnn_edges_np)

        # Optional: concat check before final Dense
        tf_concat = np.concatenate(
            [tf_gnn_nodes.reshape(B, -1), tf_gnn_edges.reshape(B, -1)], axis=-1
        )
        pt_concat = np.concatenate(
            [pt_gnn_nodes_np.reshape(B, -1), pt_gnn_edges_np.reshape(B, -1)], axis=-1
        )
        rep("Concat to final Dense", tf_concat, pt_concat, rtol=1e-6, atol=1e-7)

        print("✅ GNN intermediate parity test passed")


    def test_94_encoder_exact_match(self):
        """Test that encoder outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 94: ENCODER EXACT OUTPUT MATCH")
        print("="*70)
        
        # Transfer encoder weights
        print("Transferring encoder weights...")
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)
        
        # Use the same random input for both
        tf_input = tf.constant(self.x_test_tf+1e-9)
        tf_a = tf.constant(self.a_test_tf+1e-9)
        
        pt_input = torch.from_numpy(self.x_test_pt+1e-9)
        pt_a = torch.from_numpy(self.a_test_pt+1e-9)
        
        # Get encoder outputs
        tf_encoder_out = self.tf_model.encoder([tf_input, tf_a]).numpy()
        
        with torch.no_grad():
            pt_encoder_out = self.pt_model.encoder(pt_input, pt_a).cpu().numpy()
        
        print(f"TF encoder output shape: {tf_encoder_out.shape}")
        print(f"PT encoder output shape: {pt_encoder_out.shape}")
        print(f"TF output sample: {tf_encoder_out[0, :5]}")
        print(f"PT output sample: {pt_encoder_out[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_encoder_out - pt_encoder_out).max():.8f}")
        print(f"Mean absolute difference: {np.abs(tf_encoder_out - pt_encoder_out).mean():.8f}")
        
        # Assert close match
        np.testing.assert_allclose(
            tf_encoder_out,
            pt_encoder_out,
            rtol=1e-4,
            atol=1e-5,
            err_msg="Encoder outputs don't match!"
        )
        print("✅ Encoder outputs match!")
        print("="*70)

    def test_95_vq_layer_exact_match(self):
        """Test that VQ layer outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 95: VQ LAYER EXACT OUTPUT MATCH")
        print("="*70)
        
        # VQ weights already transferred in setUp
        # Create dummy encoder output (same for both)
        np.random.seed(123)
        encoder_out = np.random.randn(self.batch_size, self.latent_dim).astype(np.float32)
        
        # TF VQ layer
        tf_enc = tf.constant(encoder_out)
        tf_vq_layer = self.tf_model.vqvae.get_layer('vector_quantizer')
        tf_quantized, tf_soft = tf_vq_layer(tf_enc, training=False)
        
        # PT VQ layer
        pt_enc = torch.from_numpy(encoder_out)
        with torch.no_grad():
            pt_quantized, pt_soft = self.pt_model.vq_layer(pt_enc, return_losses=False)
        
        # Compare quantized outputs
        tf_quantized_np = tf_quantized.numpy()
        pt_quantized_np = pt_quantized.cpu().numpy()
        
        print(f"TF quantized shape: {tf_quantized_np.shape}")
        print(f"PT quantized shape: {pt_quantized_np.shape}")
        print(f"TF quantized sample: {tf_quantized_np[0, :5]}")
        print(f"PT quantized sample: {pt_quantized_np[0, :5]}")
        print(f"Quantized max diff: {np.abs(tf_quantized_np - pt_quantized_np).max():.8f}")
        print(f"Quantized mean diff: {np.abs(tf_quantized_np - pt_quantized_np).mean():.8f}")
        
        np.testing.assert_allclose(
            tf_quantized_np,
            pt_quantized_np,
            rtol=1e-5,
            atol=1e-6,
            err_msg="VQ quantized outputs don't match!"
        )
        
        # Compare soft counts
        tf_soft_np = tf_soft.numpy()
        pt_soft_np = pt_soft.cpu().numpy()
        
        print(f"TF soft counts shape: {tf_soft_np.shape}")
        print(f"PT soft counts shape: {pt_soft_np.shape}")
        print(f"Soft counts max diff: {np.abs(tf_soft_np - pt_soft_np).max():.8f}")
        print(f"Soft counts mean diff: {np.abs(tf_soft_np - pt_soft_np).mean():.8f}")
        
        np.testing.assert_allclose(
            tf_soft_np,
            pt_soft_np,
            rtol=1e-5,
            atol=1e-6,
            err_msg="VQ soft counts don't match!"
        )
        
        # Verify cluster assignments are identical
        tf_clusters = np.argmax(tf_soft_np, axis=1)
        pt_clusters = np.argmax(pt_soft_np, axis=1)
        match_rate = (tf_clusters == pt_clusters).mean()
        print(f"Cluster assignment match rate: {match_rate*100:.2f}%")
        
        self.assertEqual(match_rate, 1.0, "Cluster assignments should be identical!")
        print("✅ VQ layer outputs match perfectly!")
        print("="*70)

    def test_96_decoder_exact_match(self):
        """Test that decoder outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 96: DECODER EXACT OUTPUT MATCH")
        print("="*70)
        
        # Transfer decoder weights
        print("Transferring decoder weights...")
        transfer_recurrent_decoder_weights(self.tf_model.decoder, self.pt_model.decoder)
        
        # Create dummy latent code (same for both)
        np.random.seed(456)
        latent = np.random.randn(self.batch_size, self.latent_dim).astype(np.float32)
        
        # TF decoder expects flattened input for teacher forcing
        tf_latent = tf.constant(latent)
        tf_x_input = tf.constant(self.x_test_tf)
        tf_decoder_out = self.tf_model.decoder([tf_latent, tf_x_input], training=False)
        
        # PT decoder also expects flattened input
        pt_latent = torch.from_numpy(latent)
        pt_x_input = torch.from_numpy(self.x_test_pt)
        B, T, N, F = pt_x_input.shape
        pt_x_flat = pt_x_input.view(B, T, N * F)
        
        with torch.no_grad():
            pt_decoder_out = self.pt_model.decoder(pt_latent, pt_x_flat)
        
        # Handle transformer decoder which returns tuple
        if self.pt_model.encoder_type == "transformer":
            if isinstance(tf_decoder_out, tuple):
                tf_decoder_out = tf_decoder_out[0]
            if isinstance(pt_decoder_out, tuple):
                pt_decoder_out = pt_decoder_out[0]
        
        # Compare distribution means
        tf_mean = tf_decoder_out.mean().numpy()
        pt_mean = pt_decoder_out.mean.cpu().numpy()
        
        print(f"TF decoder mean shape: {tf_mean.shape}")
        print(f"PT decoder mean shape: {pt_mean.shape}")
        print(f"TF mean sample: {tf_mean[0, 0, :5]}")
        print(f"PT mean sample: {pt_mean[0, 0, :5]}")
        print(f"Decoder mean max diff: {np.abs(tf_mean - pt_mean).max():.8f}")
        print(f"Decoder mean avg diff: {np.abs(tf_mean - pt_mean).mean():.8f}")
        
        np.testing.assert_allclose(
            tf_mean,
            pt_mean,
            rtol=1e-4,
            atol=1e-5,
            err_msg="Decoder means don't match!"
        )
        
        # Compare distribution scales/stddevs
        # Note: PyTorch distributions use .scale instead of .stddev
        tf_stddev = tf_decoder_out.stddev().numpy()
        samples = pt_decoder_out.sample((10000,))
        stddev_estimate = samples.std(dim=0).cpu().numpy()  #as pt_scale = pt_decoder_out.stddev().cpu().numpy() is not implemented

        print(f"TF stddev sample: {tf_stddev[0, 0, :5]}")
        print(f"PT scale sample: {stddev_estimate[0, 0, :5]}")
        print(f"Decoder scale sum diff: {np.abs(tf_stddev.sum() - stddev_estimate.sum()):.8f}")
        
        np.testing.assert_allclose(
            tf_stddev,
            stddev_estimate,
            rtol=0.1,  #high tolerance, as only sampled
            atol=0.1,
            err_msg="Decoder scales don't match!"
        )
        print("✅ Decoder outputs match!")
        print("="*70)

    def test_97_full_model_exact_match(self):
        """Test that full VQ-VAE outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 97: FULL MODEL EXACT OUTPUT MATCH")
        print("="*70)
        
        # Transfer all weights
        print("Transferring all model weights...")
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)
        transfer_vq_weights(
            self.tf_model.vqvae.get_layer('vector_quantizer'),
            self.pt_model.vq_layer
        )
        transfer_recurrent_decoder_weights(self.tf_model.decoder, self.pt_model.decoder)
        print("All weights transferred.")
        
        # Use same random seed for both
        np.random.seed(789)
        x_test = np.random.randn(*self.input_shape_tf).astype(np.float32)
        a_test = np.random.randn(*self.edge_feature_shape_tf).astype(np.float32)
        
        # TF forward pass
        tf_input = tf.constant(x_test)
        tf_a = tf.constant(a_test)
        tf_output = self.tf_model.vqvae([tf_input, tf_a], training=False)
        
        # PT forward pass
        x_test_pt = x_test.reshape(
            self.batch_size, self.time_steps, self.num_nodes, self.n_node_features
        )
        a_test_pt = a_test.reshape(
            self.batch_size, self.time_steps, self.num_edges, self.n_edge_features
        )
        pt_input = torch.from_numpy(x_test_pt)
        pt_a = torch.from_numpy(a_test_pt)
        
        with torch.no_grad():
            pt_enc_recon, pt_recon = self.pt_model(pt_input, pt_a, return_losses=False)
        
        # Handle transformer outputs
        if self.pt_model.encoder_type == "transformer":
            if isinstance(tf_output, tuple):
                tf_output = tf_output[0]
            if isinstance(pt_enc_recon, tuple):
                pt_enc_recon = pt_enc_recon[0]
        
        # Compare encoding reconstruction (main output)
        tf_mean = tf_output.mean().numpy()
        pt_mean = pt_enc_recon.mean.cpu().numpy()
        
        print(f"TF output shape: {tf_mean.shape}")
        print(f"PT output shape: {pt_mean.shape}")
        print(f"TF output sample: {tf_mean[0, 0, :5]}")
        print(f"PT output sample: {pt_mean[0, 0, :5]}")
        print(f"Output mean max diff: {np.abs(tf_mean - pt_mean).max():.8f}")
        print(f"Output mean avg diff: {np.abs(tf_mean - pt_mean).mean():.8f}")
        
        relative_diff = np.abs(tf_mean - pt_mean) / (np.abs(tf_mean) + 1e-8)
        print(f"Output relative diff (mean): {relative_diff.mean():.8f}")
        print(f"Output relative diff (max): {relative_diff.max():.8f}")
        
        # Test encoding reconstruction
        np.testing.assert_allclose(
            tf_mean,
            pt_mean,
            rtol=1e-4,
            atol=1e-4,
            err_msg="Full model encoding reconstruction outputs don't match!"
        )
        
        # Also compare scales
        tf_stddev = tf_output.stddev().numpy()
        samples = pt_enc_recon.sample((10000,))
        pt_stddev_estimate = samples.std(dim=0).cpu().numpy()
        
        print(f"Scale sum diff: {np.abs(tf_stddev.sum() - pt_stddev_estimate.sum()):.8f}")
        
        np.testing.assert_allclose(
            tf_stddev,
            pt_stddev_estimate,
            rtol=0.1, #high tolerance as only sample
            atol=0.1,
            err_msg="Full model scales don't match!"
        )
        
        # Test bypass reconstruction (from original encoder output)
        if self.pt_model.encoder_type == "transformer" and isinstance(pt_recon, tuple):
            pt_recon = pt_recon[0]
        
        pt_recon_mean = pt_recon.mean.cpu().numpy()
        print(f"\nBypass reconstruction shape: {pt_recon_mean.shape}")
        print(f"Bypass differs from main by: {np.abs(tf_mean - pt_recon_mean).mean():.8f}")
        
        # Test that cluster assignments are identical
        print("\nTesting cluster assignments...")
        tf_soft = self.tf_model.soft_grouper([tf_input, tf_a]).numpy()
        with torch.no_grad():
            pt_soft = self.pt_model.soft_group(pt_input, pt_a).cpu().numpy()
        
        # Compare soft probabilities
        print(f"Soft probability max diff: {np.abs(tf_soft - pt_soft).max():.8f}")
        np.testing.assert_allclose(
            tf_soft,
            pt_soft,
            rtol=1e-4,
            atol=1e-5,
            err_msg="Soft cluster probabilities don't match!"
        )
        
        # Compare hard assignments
        tf_clusters = np.argmax(tf_soft, axis=1)
        pt_clusters = np.argmax(pt_soft, axis=1)
        
        cluster_match_rate = (tf_clusters == pt_clusters).mean()
        print(f"Cluster assignment match rate: {cluster_match_rate*100:.2f}%")
        
        self.assertGreaterEqual(
            cluster_match_rate,
            0.99,
            f"Cluster assignments should match at least 99%, got {cluster_match_rate*100:.2f}%"
        )
        
        # Print cluster distribution
        tf_unique, tf_counts = np.unique(tf_clusters, return_counts=True)
        pt_unique, pt_counts = np.unique(pt_clusters, return_counts=True)
        print(f"\nTF cluster distribution: {dict(zip(tf_unique, tf_counts))}")
        print(f"PT cluster distribution: {dict(zip(pt_unique, pt_counts))}")
        
        # Verify both models populate similar number of clusters
        n_tf_clusters = len(tf_unique)
        n_pt_clusters = len(pt_unique)
        print(f"TF populated clusters: {n_tf_clusters}/{self.n_components}")
        print(f"PT populated clusters: {n_pt_clusters}/{self.n_components}")
        
        self.assertEqual(
            n_tf_clusters,
            n_pt_clusters,
            "Number of populated clusters should match!"
        )
        
        print("✅ Full model outputs match!")
        print("="*70)

    @classmethod
    def tearDownClass(cls):
        """Print summary after all tests."""
        print("\n" + "="*70)
        print(" TEST SUMMARY")
        print("="*70)
        print(" ✅ All VQ-VAE translation tests completed!")
        print("="*70 + "\n")


# Run the tests
if __name__ == '__main__':
    runner = unittest.TextTestRunner(verbosity=2)
    suite = unittest.TestLoader().loadTestsFromTestCase(TestVQVAETranslation)
    result = runner.run(suite)
    
    if result.wasSuccessful():
        print(f"\n{'='*70}")
        print(f" ✅ SUCCESS: All {result.testsRun} tests passed!")
        print(f"{'='*70}")

test_000gnn_reshape_and_intermediate_parity (__main__.TestVQVAETranslation)
Compare, step by step, what TF vs PT feed into and get out of the GNN path: ... ERROR
test_1_model_structure (__main__.TestVQVAETranslation)
Test that model components exist and have correct structure. ... ok
test_2_encoder_output_shape (__main__.TestVQVAETranslation)
Test encoder output shapes match. ... ok
test_3_vq_layer_quantization (__main__.TestVQVAETranslation)
Test VQ layer produces same quantized outputs. ... ok
test_4_grouper_outputs (__main__.TestVQVAETranslation)
Test grouper (quantized latents) outputs match. ... ok
test_5_soft_grouper_outputs (__main__.TestVQVAETranslation)
Test soft_grouper (soft counts) outputs match. ... ok
test_6_forward_pass_outputs (__main__.TestVQVAETranslation)
Test full forward pass output shapes. ... ok
test_7_loss_computation (__main__.TestVQVAETranslation)
Test that losses can be computed properly. ... ok
test_8_backward_pass (__main__.TestVQVAETranslation)
Test that g